## Limpieza de datos

Eliminamos las filas que no tienen player_name o punto

In [1]:
from pathlib import Path

DATA_DIR = (Path('data') if Path('data').is_dir() else Path('../data')).resolve()

import pandas as pd
from IPython.display import display

df_base_videos = pd.read_csv(DATA_DIR / 'raw' / 'Base_Videos_Final.csv')

# Tamaño antes de la limpieza
print(f"Tamaño original: {df_base_videos.shape}")

# Eliminar filas sin player_name o punto
df_base_videos_clean = df_base_videos.dropna(subset=['player_name', 'punto'])

# Tamaño después de la limpieza
print(f"Tamaño después de limpieza: {df_base_videos_clean.shape}")
print(f"Filas eliminadas: {len(df_base_videos) - len(df_base_videos_clean)}")

# Actualizar el dataframe principal
df_base_videos = df_base_videos_clean.copy()

NameError: name 'df_base_videos' is not defined

## Eliminación de jugadores genéricos

Eliminamos las filas donde el nombre del jugador es Player_x (jugadores no identificados)

In [ ]:
# Verificar cuántos registros tienen Player_x como nombre
player_x_count = df_base_videos[df_base_videos['player_name'].str.contains('Player_', na=False)].shape[0]
print(f"Registros con Player_x: {player_x_count}")

# Tamaño antes de eliminar
print(f"Tamaño antes: {df_base_videos.shape}")

# Eliminar filas donde player_name contenga 'Player_'
df_base_videos = df_base_videos[~df_base_videos['player_name'].str.contains('Player_', na=False)]

# Tamaño después
print(f"Tamaño después: {df_base_videos.shape}")
print(f"Filas eliminadas: {player_x_count}")

Registros con Player_x: 192358
Tamaño antes: (519424, 30)
Tamaño después: (327066, 30)
Filas eliminadas: 192358


## Renombrar columna team a posicion_cancha

La columna team actualmente solo indica "top" o "bottom", así que la renombramos a posicion_cancha

In [ ]:
# Renombrar la columna team a posicion_cancha
df_base_videos = df_base_videos.rename(columns={'team': 'posicion_cancha'})

# Verificar el cambio
display(df_base_videos[['player_name', 'posicion_cancha', 'partido', 'punto']].head(10))

,player_name,posicion_cancha,partido,punto
0,Luis Jorge Garcia Camargo,top,Partido 27,10.0
1,Luis Jorge Garcia Camargo,top,Partido 27,10.0
2,Luis Jorge Garcia Camargo,top,Partido 27,10.0
3,Luis Jorge Garcia Camargo,top,Partido 27,10.0
4,Luis Jorge Garcia Camargo,top,Partido 27,10.0
5,Luis Jorge Garcia Camargo,top,Partido 27,10.0
6,Luis Jorge Garcia Camargo,NaN,Partido 27,10.0
7,Luis Jorge Garcia Camargo,NaN,Partido 27,10.0
8,Luis Jorge Garcia Camargo,NaN,Partido 27,10.0
9,Luis Jorge Garcia Camargo,NaN,Partido 27,10.0


## Carga del archivo de levantamiento de datos

Cargamos el archivo que contiene la información de equipos y posiciones de los jugadores

In [ ]:
# Cargar el archivo de levantamiento de datos
df_levantamiento = pd.read_csv(DATA_DIR / 'raw' / 'Levantamiento_datos_proyecto.csv', sep=';', encoding='latin1')

# Mostrar información básica
display(df_levantamiento.head())
print(f"\nForma del dataset: {df_levantamiento.shape}")
display(df_levantamiento.columns.tolist())

,ID_PARTIDO,CANCHA,ID_PUNTO,ID_JUGADOR_1_EQUIPO_1,ID_JUGADOR_2_EQUIPO_1,ID_JUGADOR_1_EQUIPO_2,ID_JUGADOR_2_EQUIPO_2,POSICION_JUGADOR_1_EQUIPO_1,POSICION_JUGADOR_2_EQUIPO_1,POSICION_JUGADOR_1_EQUIPO_2,POSICION_JUGADOR_2_EQUIPO_2,PUNTO_EQUIPO_1,PUNTO_EQUIPO_2
0,27,1,1,Nicole Lizarazo,Valentina Alejandra Lopez Romero,Luis Jorge Garcia Camargo,Nikol sofia avendaño,IZQUIERDA,DERECHA,IZQUIERDA,DERECHA,1,0.0
1,27,1,2,Nicole Lizarazo,Valentina Alejandra Lopez Romero,Luis Jorge Garcia Camargo,Nikol sofia avendaño,IZQUIERDA,DERECHA,IZQUIERDA,DERECHA,0,1.0
2,27,1,3,Nicole Lizarazo,Valentina Alejandra Lopez Romero,Luis Jorge Garcia Camargo,Nikol sofia avendaño,IZQUIERDA,DERECHA,IZQUIERDA,DERECHA,0,1.0
3,27,1,4,Nicole Lizarazo,Valentina Alejandra Lopez Romero,Luis Jorge Garcia Camargo,Nikol sofia avendaño,IZQUIERDA,DERECHA,IZQUIERDA,DERECHA,1,0.0
4,27,1,5,Nicole Lizarazo,Valentina Alejandra Lopez Romero,Luis Jorge Garcia Camargo,Nikol sofia avendaño,IZQUIERDA,DERECHA,IZQUIERDA,DERECHA,0,1.0



Forma del dataset: (471, 13)


['ID_PARTIDO',
 'CANCHA',
 'ID_PUNTO',
 'ID_JUGADOR_1_EQUIPO_1',
 'ID_JUGADOR_2_EQUIPO_1',
 'ID_JUGADOR_1_EQUIPO_2',
 'ID_JUGADOR_2_EQUIPO_2',
 'POSICION_JUGADOR_1_EQUIPO_1',
 'POSICION_JUGADOR_2_EQUIPO_1',
 'POSICION_JUGADOR_1_EQUIPO_2',
 'POSICION_JUGADOR_2_EQUIPO_2',
 'PUNTO_EQUIPO_1',
 'PUNTO_EQUIPO_2']

## Normalización de nombres de jugadores

Normalizamos los nombres en ambos datasets para poder hacer el merge correctamente

In [ ]:
import unicodedata

def normalizar_nombre(nombre):
    """Normaliza un nombre eliminando acentos, convirtiendo a minúsculas y eliminando espacios extra"""
    if pd.isna(nombre):
        return nombre
    # Convertir a string y eliminar espacios extra
    nombre = str(nombre).strip()
    # Eliminar acentos
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    # Convertir a minúsculas
    nombre = nombre.lower()
    return nombre

# Normalizar nombres en df_base_videos
df_base_videos['player_name_normalized'] = df_base_videos['player_name'].apply(normalizar_nombre)

# Normalizar nombres en df_levantamiento
for col in ['ID_JUGADOR_1_EQUIPO_1', 'ID_JUGADOR_2_EQUIPO_1', 'ID_JUGADOR_1_EQUIPO_2', 'ID_JUGADOR_2_EQUIPO_2']:
    df_levantamiento[f'{col}_normalized'] = df_levantamiento[col].apply(normalizar_nombre)

# Verificar algunos ejemplos
display(df_base_videos[['player_name', 'player_name_normalized']].drop_duplicates().head(10))

,player_name,player_name_normalized
0,Luis Jorge Garcia Camargo,luis jorge garcia camargo
625,Nicole Lizarazo,nicole lizarazo
1240,Nikol sofia avendaño,nikol sofia avendano
1865,Valentina Alejandra Lopez Romero,valentina alejandra lopez romero
23839,Andrea Julieth Sosa Rodriguez,andrea julieth sosa rodriguez
25939,Valentina Jimenez Gomez,valentina jimenez gomez
43205,Jorge Enrique Lugo Lopez,jorge enrique lugo lopez
43641,María José Cifuentes,maria jose cifuentes
44513,Santiago Peña Beltran,santiago pena beltran
59801,Carlos Felipe Andrade Sáenz,carlos felipe andrade saenz


## Normalización del número de partido

En df_base_videos el partido puede estar como "Partido x" mientras que en levantamiento solo es el número

In [ ]:
# Verificar formato actual de partido
display(df_base_videos['partido'].unique()[:10])

# Función para extraer el número del partido
def extraer_numero_partido(partido):
    if pd.isna(partido):
        return partido
    # Convertir a string y normalizar
    partido_str = str(partido).strip().lower()
    # Si contiene "partido", extraer el número
    if 'partido' in partido_str:
        # Extraer solo los dígitos
        import re
        numeros = re.findall(r'\d+', partido_str)
        if numeros:
            return int(numeros[0])
    # Si ya es número, devolverlo como int
    try:
        return int(partido)
    except:
        return None

# Normalizar partido en df_base_videos
df_base_videos['partido_num'] = df_base_videos['partido'].apply(extraer_numero_partido)

# Verificar el resultado
display(df_base_videos[['partido', 'partido_num']].drop_duplicates())

array(['Partido 27', 'Partido 29', 'Partido 35', '1', '11', 11, 13, 15,
       17, 19], dtype=object)

,partido,partido_num
0,Partido 27,27
23839,Partido 29,29
43205,Partido 35,35
59801,1,1
63395,11,11
65536,11,11
89332,13,13
112671,15,15
144563,17,17
163981,19,19


## Crear mapeo de jugador-equipo-posición

Creamos un diccionario que mapee cada jugador a su equipo y posición en cancha por partido y punto

In [ ]:
# Crear un dataframe long format con la información de equipos
jugadores_info = []

for _, row in df_levantamiento.iterrows():
    partido = row['ID_PARTIDO']
    punto = row['ID_PUNTO']
    
    # Equipo 1
    jugadores_info.append({
        'partido_num': partido,
        'punto': punto,
        'player_name_normalized': row['ID_JUGADOR_1_EQUIPO_1_normalized'],
        'equipo': f'match_{partido}_team_1',
        'posicion_en_equipo': row['POSICION_JUGADOR_1_EQUIPO_1']
    })
    jugadores_info.append({
        'partido_num': partido,
        'punto': punto,
        'player_name_normalized': row['ID_JUGADOR_2_EQUIPO_1_normalized'],
        'equipo': f'match_{partido}_team_1',
        'posicion_en_equipo': row['POSICION_JUGADOR_2_EQUIPO_1']
    })
    
    # Equipo 2
    jugadores_info.append({
        'partido_num': partido,
        'punto': punto,
        'player_name_normalized': row['ID_JUGADOR_1_EQUIPO_2_normalized'],
        'equipo': f'match_{partido}_team_2',
        'posicion_en_equipo': row['POSICION_JUGADOR_1_EQUIPO_2']
    })
    jugadores_info.append({
        'partido_num': partido,
        'punto': punto,
        'player_name_normalized': row['ID_JUGADOR_2_EQUIPO_2_normalized'],
        'equipo': f'match_{partido}_team_2',
        'posicion_en_equipo': row['POSICION_JUGADOR_2_EQUIPO_2']
    })

df_jugadores_equipos = pd.DataFrame(jugadores_info)

# Verificar el resultado
display(df_jugadores_equipos.head(20))

,partido_num,punto,player_name_normalized,equipo,posicion_en_equipo
0,27,1,nicole lizarazo,match_27_team_1,IZQUIERDA
1,27,1,valentina alejandra lopez romero,match_27_team_1,DERECHA
2,27,1,luis jorge garcia camargo,match_27_team_2,IZQUIERDA
3,27,1,nikol sofia avendano,match_27_team_2,DERECHA
4,27,2,nicole lizarazo,match_27_team_1,IZQUIERDA
5,27,2,valentina alejandra lopez romero,match_27_team_1,DERECHA
6,27,2,luis jorge garcia camargo,match_27_team_2,IZQUIERDA
7,27,2,nikol sofia avendano,match_27_team_2,DERECHA
8,27,3,nicole lizarazo,match_27_team_1,IZQUIERDA
9,27,3,valentina alejandra lopez romero,match_27_team_1,DERECHA


## Hacer merge con el dataset principal

Unimos la información de equipos con el dataset principal usando partido, punto y nombre normalizado

In [ ]:
# Hacer el merge
df_base_videos = df_base_videos.merge(
    df_jugadores_equipos,
    on=['partido_num', 'punto', 'player_name_normalized'],
    how='left'
)

# Verificar el resultado del merge
print(f"Forma después del merge: {df_base_videos.shape}")
print(f"\nValores únicos en equipo:")
display(df_base_videos['equipo'].value_counts())
print(f"\nRegistros sin equipo asignado: {df_base_videos['equipo'].isna().sum()}")

# Mostrar algunos ejemplos
display(df_base_videos[['player_name', 'equipo', 'posicion_en_equipo', 'posicion_cancha', 'partido_num', 'punto']].head(20))

Forma después del merge: (327066, 34)

Valores únicos en equipo:


equipo
match_9_team_2     13074
match_18_team_2    10527
match_18_team_1    10477
match_15_team_2    10251
match_20_team_2    10011
match_27_team_2     9826
match_27_team_1     9781
match_22_team_2     9197
match_22_team_1     8997
match_29_team_1     8951
match_29_team_2     8789
match_14_team_2     7912
match_3_team_1      7558
match_13_team_2     7536
match_36_team_1     7458
match_36_team_2     7373
match_11_team_2     7306
match_14_team_1     7200
match_28_team_1     7054
match_38_team_2     6748
match_28_team_2     6693
match_38_team_1     6675
match_12_team_1     6466
match_12_team_2     6374
match_19_team_2     6332
match_37_team_1     5963
match_35_team_1     5932
match_35_team_2     5931
match_17_team_2     5889
match_37_team_2     5662
match_21_team_2     5590
match_31_team_1     4803
match_31_team_2     4752
match_20_team_1     4606
match_23_team_2     4528
match_34_team_2     4445
match_41_team_1     4105
match_41_team_2     4052
match_1_team_2      3594
match_11_team_1   


Registros sin equipo asignado: 5837


,player_name,equipo,posicion_en_equipo,posicion_cancha,partido_num,punto
0,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
1,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
2,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
3,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
4,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
5,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,top,27,10.0
6,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,NaN,27,10.0
7,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,NaN,27,10.0
8,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,NaN,27,10.0
9,Luis Jorge Garcia Camargo,match_27_team_2,IZQUIERDA,NaN,27,10.0


## Imputar valores faltantes de equipo y posición

Para registros sin equipo/posición, imputamos con el mismo valor de otro registro del mismo jugador y partido

In [ ]:
# Para cada jugador en cada partido, el equipo y posición son constantes
# Usamos forward fill y backward fill por grupo de jugador-partido

print(f"Registros sin equipo antes de imputar: {df_base_videos['equipo'].isna().sum()}")
print(f"Registros sin posicion_en_equipo antes de imputar: {df_base_videos['posicion_en_equipo'].isna().sum()}")
print(f"Registros sin posicion_cancha antes de imputar: {df_base_videos['posicion_cancha'].isna().sum()}")

# Imputar equipo y posicion_en_equipo usando los valores conocidos del mismo jugador en el mismo partido
df_base_videos['equipo'] = df_base_videos.groupby(['player_name_normalized', 'partido_num'])['equipo'].transform(
    lambda x: x.ffill().bfill()
)

df_base_videos['posicion_en_equipo'] = df_base_videos.groupby(['player_name_normalized', 'partido_num'])['posicion_en_equipo'].transform(
    lambda x: x.ffill().bfill()
)

df_base_videos['posicion_cancha'] = df_base_videos.groupby(['player_name_normalized', 'partido_num'])['posicion_cancha'].transform(
    lambda x: x.ffill().bfill()
)

print(f"\nRegistros sin equipo después de imputar: {df_base_videos['equipo'].isna().sum()}")
print(f"Registros sin posicion_en_equipo después de imputar: {df_base_videos['posicion_en_equipo'].isna().sum()}")
print(f"Registros sin posicion_cancha después de imputar: {df_base_videos['posicion_cancha'].isna().sum()}")

# Mostrar resumen
display(df_base_videos[['player_name', 'equipo', 'posicion_en_equipo', 'posicion_cancha']].describe())

Registros sin equipo antes de imputar: 5837
Registros sin posicion_en_equipo antes de imputar: 5837
Registros sin posicion_cancha antes de imputar: 193306

Registros sin equipo después de imputar: 0
Registros sin posicion_en_equipo después de imputar: 0
Registros sin posicion_cancha después de imputar: 33842


,player_name,equipo,posicion_en_equipo,posicion_cancha
count,327066,327066,327066,293224
unique,36,54,2,2
top,María José Cifuentes,match_9_team_2,IZQUIERDA,bottom
freq,18532,13074,172802,151807


## Resumen final del procesamiento

Verificamos el estado final del dataset con todas las transformaciones aplicadas

In [ ]:
# Resumen del dataset final
print("="*60)
print("RESUMEN DEL DATASET FINAL")
print("="*60)
print(f"\nForma del dataset: {df_base_videos.shape}")
print(f"\nColumnas principales:")
display(['player_name', 'equipo', 'posicion_en_equipo', 'posicion_cancha', 'partido_num', 'punto'])

print(f"\n\nEstadísticas de equipos:")
display(df_base_videos.groupby('equipo')[['player_name']].nunique().rename(columns={'player_name': 'num_jugadores'}))

print(f"\n\nEstadísticas de posiciones:")
print(f"Posiciones en equipo:")
display(df_base_videos['posicion_en_equipo'].value_counts())
print(f"\nPosiciones en cancha:")
display(df_base_videos['posicion_cancha'].value_counts())

print(f"\n\nJugadores únicos: {df_base_videos['player_name'].nunique()}")
print(f"Partidos únicos: {df_base_videos['partido_num'].nunique()}")
print(f"Equipos únicos: {df_base_videos['equipo'].nunique()}")

RESUMEN DEL DATASET FINAL

Forma del dataset: (327066, 34)

Columnas principales:


['player_name',
 'equipo',
 'posicion_en_equipo',
 'posicion_cancha',
 'partido_num',
 'punto']



Estadísticas de equipos:


,num_jugadores
equipo,
match_11_team_1,1
match_11_team_2,2
match_12_team_1,2
match_12_team_2,2
match_13_team_2,2
match_14_team_1,2
match_14_team_2,2
match_15_team_2,2
match_17_team_2,2




Estadísticas de posiciones:
Posiciones en equipo:


posicion_en_equipo
IZQUIERDA    172802
DERECHA      154264
Name: count, dtype: int64


Posiciones en cancha:


posicion_cancha
bottom    151807
top       141417
Name: count, dtype: int64



Jugadores únicos: 36
Partidos únicos: 32
Equipos únicos: 54


## Verificación de valores nulos después del procesamiento

Revisamos nuevamente qué columnas aún tienen valores nulos después de todas las transformaciones

In [ ]:
# Calcular el porcentaje de valores nulos por columna
porcentaje_nulos = (df_base_videos.isnull().sum() / len(df_base_videos) * 100).sort_values(ascending=False)
porcentaje_nulos_df = pd.DataFrame({
    'Columna': porcentaje_nulos.index,
    'Porcentaje_Nulos': porcentaje_nulos.values
})

display(porcentaje_nulos_df[porcentaje_nulos_df['Porcentaje_Nulos'] > 0])

,Columna,Porcentaje_Nulos
0,time_since_last_hit,97.782405
1,distance_player_to_teammate_m,40.517204
2,ball_speed_mps,31.899372
3,ball_displacement,31.899372
4,ball_position_x_prev,31.899372
5,ball_position_y_prev,31.899372
6,posicion_cancha,10.347147
7,prev_speed,0.621281
8,player_acceleration_mps2,0.621281
9,player_speed_mps,0.310641


## Guardar dataset procesado

Guardamos el dataframe procesado con todas las transformaciones aplicadas

In [ ]:
# Guardar el dataframe procesado en un archivo CSV
df_base_videos.to_csv(DATA_DIR / 'interim' / 'Base_Videos_Final_Procesado.csv', index=False)

## Imputación de distance_player_to_teammate_m

Utilizamos interpolación temporal por partido y equipo para imputar los valores faltantes de distancia entre compañeros

In [ ]:
# Verificar valores nulos antes de la imputación
print(f"Valores nulos en distance_player_to_teammate_m antes: {df_base_videos['distance_player_to_teammate_m'].isna().sum()}")
print(f"Porcentaje: {df_base_videos['distance_player_to_teammate_m'].isna().sum() / len(df_base_videos) * 100:.2f}%")

# Ordenar por partido, equipo y frame para interpolación temporal correcta
df_base_videos = df_base_videos.sort_values(['partido_num', 'equipo', 'frame_idx'])

# Aplicar interpolación temporal por grupo de partido y equipo
df_base_videos['distance_player_to_teammate_m'] = df_base_videos.groupby(['partido_num', 'equipo'])['distance_player_to_teammate_m'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

# Verificar valores nulos después de la imputación
print(f"\nValores nulos en distance_player_to_teammate_m después: {df_base_videos['distance_player_to_teammate_m'].isna().sum()}")
print(f"Porcentaje: {df_base_videos['distance_player_to_teammate_m'].isna().sum() / len(df_base_videos) * 100:.2f}%")

# Mostrar estadísticas de la variable imputada
display(df_base_videos['distance_player_to_teammate_m'].describe())

Valores nulos en distance_player_to_teammate_m antes: 132518
Porcentaje: 40.52%

Valores nulos en distance_player_to_teammate_m después: 0
Porcentaje: 0.00%


count    327066.000000
mean          5.300188
std           2.125062
min           0.000000
25%           4.136854
50%           5.474157
75%           6.526783
max          23.147162
Name: distance_player_to_teammate_m, dtype: float64

## Imputación de variables de trayectoria de la pelota

Interpolamos las posiciones de la pelota y recalculamos las variables derivadas (prev, displacement, speed) respetando límites físicos

## Paso 1: Verificación inicial de variables de la pelota

Revisamos el estado de las variables de trayectoria de la pelota antes de interpolar

In [ ]:
# Paso 1: Verificar valores nulos antes de la interpolación
print("="*60)
print("IMPUTACIÓN DE VARIABLES DE TRAYECTORIA DE LA PELOTA")
print("="*60)
print("\nValores nulos ANTES de la interpolación:")
print(f"ball_position_x: {df_base_videos['ball_position_x'].isna().sum()} ({df_base_videos['ball_position_x'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"ball_position_y: {df_base_videos['ball_position_y'].isna().sum()} ({df_base_videos['ball_position_y'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"ball_position_x_prev: {df_base_videos['ball_position_x_prev'].isna().sum()} ({df_base_videos['ball_position_x_prev'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"ball_position_y_prev: {df_base_videos['ball_position_y_prev'].isna().sum()} ({df_base_videos['ball_position_y_prev'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"ball_displacement: {df_base_videos['ball_displacement'].isna().sum()} ({df_base_videos['ball_displacement'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"ball_speed_mps: {df_base_videos['ball_speed_mps'].isna().sum()} ({df_base_videos['ball_speed_mps'].isna().sum() / len(df_base_videos) * 100:.2f}%)")

IMPUTACIÓN DE VARIABLES DE TRAYECTORIA DE LA PELOTA

Valores nulos ANTES de la interpolación:
ball_position_x: 0 (0.00%)
ball_position_y: 0 (0.00%)
ball_position_x_prev: 104332 (31.90%)
ball_position_y_prev: 104332 (31.90%)
ball_displacement: 104332 (31.90%)
ball_speed_mps: 104332 (31.90%)


## Paso 2: Interpolación de posiciones de la pelota

Interpolamos linealmente las posiciones x e y de la pelota por partido y punto

In [ ]:
# Paso 2: Interpolar las posiciones de la pelota por partido y punto
print("\n" + "="*60)
print("PASO 1: INTERPOLACIÓN DE POSICIONES")
print("="*60)

# Asegurar que está ordenado por partido, punto y frame
df_base_videos = df_base_videos.sort_values(['partido_num', 'punto', 'frame_idx'])

# Interpolar ball_position_x y ball_position_y por grupo de partido y punto
df_base_videos['ball_position_x'] = df_base_videos.groupby(['partido_num', 'punto'])['ball_position_x'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

df_base_videos['ball_position_y'] = df_base_videos.groupby(['partido_num', 'punto'])['ball_position_y'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

print(f"\nValores nulos después de interpolar posiciones:")
print(f"ball_position_x: {df_base_videos['ball_position_x'].isna().sum()}")
print(f"ball_position_y: {df_base_videos['ball_position_y'].isna().sum()}")


PASO 1: INTERPOLACIÓN DE POSICIONES

Valores nulos después de interpolar posiciones:
ball_position_x: 0
ball_position_y: 0


## Paso 3: Recalcular posiciones previas de la pelota

Calculamos las posiciones previas usando shift para determinar el movimiento de la pelota

In [ ]:
# Paso 3: Recalcular las posiciones previas con shift por partido y punto
print("\n" + "="*60)
print("PASO 2: RECALCULAR POSICIONES PREVIAS")
print("="*60)

# Calcular las posiciones previas usando shift por grupo
df_base_videos['ball_position_x_prev'] = df_base_videos.groupby(['partido_num', 'punto'])['ball_position_x'].shift(1)
df_base_videos['ball_position_y_prev'] = df_base_videos.groupby(['partido_num', 'punto'])['ball_position_y'].shift(1)

# Rellenar los primeros valores de cada grupo (que quedan NaN) con 0
df_base_videos['ball_position_x_prev'] = df_base_videos['ball_position_x_prev'].fillna(0)
df_base_videos['ball_position_y_prev'] = df_base_videos['ball_position_y_prev'].fillna(0)

print(f"\nValores nulos después de recalcular prev:")
print(f"ball_position_x_prev: {df_base_videos['ball_position_x_prev'].isna().sum()}")
print(f"ball_position_y_prev: {df_base_videos['ball_position_y_prev'].isna().sum()}")


PASO 2: RECALCULAR POSICIONES PREVIAS

Valores nulos después de recalcular prev:
ball_position_x_prev: 0
ball_position_y_prev: 0


## Paso 4: Calcular desplazamiento de la pelota

Calculamos el desplazamiento de la pelota usando las distancias euclidianas entre posiciones

In [ ]:
# Paso 4: Recalcular el desplazamiento de la pelota
print("\n" + "="*60)
print("PASO 3: RECALCULAR DISPLACEMENT")
print("="*60)

import numpy as np

# Calcular el desplazamiento usando las posiciones actuales y previas
df_base_videos['ball_displacement'] = np.sqrt(
    (df_base_videos['ball_position_x'] - df_base_videos['ball_position_x_prev'])**2 +
    (df_base_videos['ball_position_y'] - df_base_videos['ball_position_y_prev'])**2
)

print(f"\nValores nulos después de recalcular displacement:")
print(f"ball_displacement: {df_base_videos['ball_displacement'].isna().sum()}")
print(f"\nEstadísticas de ball_displacement:")
display(df_base_videos['ball_displacement'].describe())


PASO 3: RECALCULAR DISPLACEMENT

Valores nulos después de recalcular displacement:
ball_displacement: 0

Estadísticas de ball_displacement:


count    327066.000000
mean          0.064861
std           0.553454
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          26.778434
Name: ball_displacement, dtype: float64

## Paso 5: Calcular velocidad de la pelota con límites físicos

Calculamos la velocidad de la pelota y aplicamos un límite máximo realista de 20 m/s

In [ ]:
# Paso 5: Recalcular la velocidad de la pelota con límites físicos
print("\n" + "="*60)
print("PASO 4: RECALCULAR VELOCIDAD CON LÍMITES FÍSICOS")
print("="*60)

# Calcular velocidad: displacement / duration
# Asumiendo que duration está en segundos
df_base_videos['ball_speed_mps'] = df_base_videos['ball_displacement'] / df_base_videos['duration']

# Aplicar límites físicos:
# Velocidad máxima de smash: ~20 m/s (72 km/h)
# Velocidad promedio: ~10-15 m/s
MAX_BALL_SPEED = 20.0  # m/s

# Identificar valores que exceden el límite
valores_exceden = (df_base_videos['ball_speed_mps'] > MAX_BALL_SPEED).sum()
print(f"\nValores que exceden {MAX_BALL_SPEED} m/s: {valores_exceden} ({valores_exceden / len(df_base_videos) * 100:.2f}%)")

# Limitar velocidades al máximo físico
df_base_videos['ball_speed_mps'] = df_base_videos['ball_speed_mps'].clip(upper=MAX_BALL_SPEED)

print(f"\nValores nulos después de recalcular speed:")
print(f"ball_speed_mps: {df_base_videos['ball_speed_mps'].isna().sum()}")
print(f"\nEstadísticas de ball_speed_mps (después de límite):")
display(df_base_videos['ball_speed_mps'].describe())


PASO 4: RECALCULAR VELOCIDAD CON LÍMITES FÍSICOS

Valores que exceden 20.0 m/s: 7652 (2.34%)

Valores nulos después de recalcular speed:
ball_speed_mps: 0

Estadísticas de ball_speed_mps (después de límite):


count    327066.000000
mean          1.196692
std           3.780181
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          20.000000
Name: ball_speed_mps, dtype: float64

## Resumen final: Variables de trayectoria de la pelota

Resumen de todas las transformaciones y estadísticas finales de las variables de la pelota

In [ ]:
# Resumen final de la imputación de variables de la pelota
print("\n" + "="*60)
print("RESUMEN FINAL - VARIABLES DE TRAYECTORIA DE LA PELOTA")
print("="*60)

print("\nValores nulos finales:")
print(f"ball_position_x: {df_base_videos['ball_position_x'].isna().sum()}")
print(f"ball_position_y: {df_base_videos['ball_position_y'].isna().sum()}")
print(f"ball_position_x_prev: {df_base_videos['ball_position_x_prev'].isna().sum()}")
print(f"ball_position_y_prev: {df_base_videos['ball_position_y_prev'].isna().sum()}")
print(f"ball_displacement: {df_base_videos['ball_displacement'].isna().sum()}")
print(f"ball_speed_mps: {df_base_videos['ball_speed_mps'].isna().sum()}")

print("\n\nEstadísticas finales de ball_speed_mps:")
display(df_base_videos['ball_speed_mps'].describe())

print(f"\n\nRango de velocidades:")
print(f"Mínima: {df_base_videos['ball_speed_mps'].min():.2f} m/s ({df_base_videos['ball_speed_mps'].min() * 3.6:.2f} km/h)")
print(f"Máxima: {df_base_videos['ball_speed_mps'].max():.2f} m/s ({df_base_videos['ball_speed_mps'].max() * 3.6:.2f} km/h)")
print(f"Media: {df_base_videos['ball_speed_mps'].mean():.2f} m/s ({df_base_videos['ball_speed_mps'].mean() * 3.6:.2f} km/h)")
print(f"Mediana: {df_base_videos['ball_speed_mps'].median():.2f} m/s ({df_base_videos['ball_speed_mps'].median() * 3.6:.2f} km/h)")


RESUMEN FINAL - VARIABLES DE TRAYECTORIA DE LA PELOTA

Valores nulos finales:
ball_position_x: 0
ball_position_y: 0
ball_position_x_prev: 0
ball_position_y_prev: 0
ball_displacement: 0
ball_speed_mps: 0


Estadísticas finales de ball_speed_mps:


count    327066.000000
mean          1.196692
std           3.780181
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          20.000000
Name: ball_speed_mps, dtype: float64



Rango de velocidades:
Mínima: 0.00 m/s (0.00 km/h)
Máxima: 20.00 m/s (72.00 km/h)
Media: 1.20 m/s (4.31 km/h)
Mediana: 0.00 m/s (0.00 km/h)


## Imputación de variables de desplazamiento del jugador

Interpolamos las posiciones del jugador y recalculamos las variables derivadas (prev, displacement, speed, acceleration) respetando límites físicos

## Paso 1: Verificación inicial de variables del jugador

Revisamos el estado de las variables de movimiento del jugador antes de interpolar

In [ ]:
# Paso 1: Verificar valores nulos antes de la interpolación
print("="*60)
print("IMPUTACIÓN DE VARIABLES DE DESPLAZAMIENTO DEL JUGADOR")
print("="*60)
print("\nValores nulos ANTES de la interpolación:")
print(f"player_position_x: {df_base_videos['player_position_x'].isna().sum()} ({df_base_videos['player_position_x'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"player_position_y: {df_base_videos['player_position_y'].isna().sum()} ({df_base_videos['player_position_y'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"prev_x: {df_base_videos['prev_x'].isna().sum()} ({df_base_videos['prev_x'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"prev_y: {df_base_videos['prev_y'].isna().sum()} ({df_base_videos['prev_y'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"player_displacement: {df_base_videos['player_displacement'].isna().sum()} ({df_base_videos['player_displacement'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"player_speed_mps: {df_base_videos['player_speed_mps'].isna().sum()} ({df_base_videos['player_speed_mps'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"prev_speed: {df_base_videos['prev_speed'].isna().sum()} ({df_base_videos['prev_speed'].isna().sum() / len(df_base_videos) * 100:.2f}%)")
print(f"player_acceleration_mps2: {df_base_videos['player_acceleration_mps2'].isna().sum()} ({df_base_videos['player_acceleration_mps2'].isna().sum() / len(df_base_videos) * 100:.2f}%)")

IMPUTACIÓN DE VARIABLES DE DESPLAZAMIENTO DEL JUGADOR

Valores nulos ANTES de la interpolación:
player_position_x: 0 (0.00%)
player_position_y: 0 (0.00%)
prev_x: 1016 (0.31%)
prev_y: 1016 (0.31%)
player_displacement: 1016 (0.31%)
player_speed_mps: 1016 (0.31%)
prev_speed: 2032 (0.62%)
player_acceleration_mps2: 2032 (0.62%)


## Paso 2: Interpolación de posiciones del jugador

Interpolamos linealmente las posiciones x e y del jugador por partido, jugador y punto

In [ ]:
# Paso 2: Interpolar las posiciones del jugador por partido, jugador y punto
print("\n" + "="*60)
print("PASO 1: INTERPOLACIÓN DE POSICIONES DEL JUGADOR")
print("="*60)

# Interpolar player_position_x y player_position_y por grupo de partido, jugador y punto
df_base_videos['player_position_x'] = df_base_videos.groupby(['partido_num', 'player_name_normalized', 'punto'])['player_position_x'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

df_base_videos['player_position_y'] = df_base_videos.groupby(['partido_num', 'player_name_normalized', 'punto'])['player_position_y'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

print(f"\nValores nulos después de interpolar posiciones:")
print(f"player_position_x: {df_base_videos['player_position_x'].isna().sum()}")
print(f"player_position_y: {df_base_videos['player_position_y'].isna().sum()}")


PASO 1: INTERPOLACIÓN DE POSICIONES DEL JUGADOR

Valores nulos después de interpolar posiciones:
player_position_x: 0
player_position_y: 0


## Paso 3: Recalcular posiciones previas del jugador

Calculamos las posiciones previas del jugador usando shift para determinar su movimiento

In [ ]:
# Paso 3: Recalcular las posiciones previas con shift por partido, jugador y punto
print("\n" + "="*60)
print("PASO 2: RECALCULAR POSICIONES PREVIAS DEL JUGADOR")
print("="*60)

# Calcular las posiciones previas usando shift por grupo
df_base_videos['prev_x'] = df_base_videos.groupby(['partido_num', 'player_name_normalized', 'punto'])['player_position_x'].shift(1)
df_base_videos['prev_y'] = df_base_videos.groupby(['partido_num', 'player_name_normalized', 'punto'])['player_position_y'].shift(1)

# Rellenar los primeros valores de cada grupo (que quedan NaN) con 0
df_base_videos['prev_x'] = df_base_videos['prev_x'].fillna(0)
df_base_videos['prev_y'] = df_base_videos['prev_y'].fillna(0)

print(f"\nValores nulos después de recalcular prev:")
print(f"prev_x: {df_base_videos['prev_x'].isna().sum()}")
print(f"prev_y: {df_base_videos['prev_y'].isna().sum()}")


PASO 2: RECALCULAR POSICIONES PREVIAS DEL JUGADOR

Valores nulos después de recalcular prev:
prev_x: 0
prev_y: 0


## Paso 4: Calcular desplazamiento del jugador

Calculamos el desplazamiento del jugador usando las distancias euclidianas entre posiciones

In [ ]:
# Paso 4: Recalcular el desplazamiento del jugador
print("\n" + "="*60)
print("PASO 3: RECALCULAR DISPLACEMENT DEL JUGADOR")
print("="*60)

# Calcular el desplazamiento usando las posiciones actuales y previas
df_base_videos['player_displacement'] = np.sqrt(
    (df_base_videos['player_position_x'] - df_base_videos['prev_x'])**2 +
    (df_base_videos['player_position_y'] - df_base_videos['prev_y'])**2
)

print(f"\nValores nulos después de recalcular displacement:")
print(f"player_displacement: {df_base_videos['player_displacement'].isna().sum()}")
print(f"\nEstadísticas de player_displacement:")
display(df_base_videos['player_displacement'].describe())


PASO 3: RECALCULAR DISPLACEMENT DEL JUGADOR

Valores nulos después de recalcular displacement:
player_displacement: 0

Estadísticas de player_displacement:


count    327066.000000
mean          0.063759
std           0.562918
min           0.000000
25%           0.000000
50%           0.000000
75%           0.056569
max          17.988797
Name: player_displacement, dtype: float64

## Paso 5: Calcular velocidad del jugador con límites físicos

Calculamos la velocidad del jugador y aplicamos un límite máximo realista de 4.17 m/s (15 km/h)

In [ ]:
# Paso 5: Recalcular la velocidad del jugador con límites físicos
print("\n" + "="*60)
print("PASO 4: RECALCULAR VELOCIDAD DEL JUGADOR CON LÍMITES FÍSICOS")
print("="*60)

# Calcular velocidad: displacement / duration
df_base_videos['player_speed_mps'] = df_base_videos['player_displacement'] / df_base_videos['duration']

# Aplicar límites físicos:
# Velocidad máxima del jugador: ~15 km/h = ~4.17 m/s
MAX_PLAYER_SPEED = 4.17  # m/s (≈15 km/h)

# Identificar valores que exceden el límite
valores_exceden_speed = (df_base_videos['player_speed_mps'] > MAX_PLAYER_SPEED).sum()
print(f"\nValores que exceden {MAX_PLAYER_SPEED} m/s: {valores_exceden_speed} ({valores_exceden_speed / len(df_base_videos) * 100:.2f}%)")

# Limitar velocidades al máximo físico
df_base_videos['player_speed_mps'] = df_base_videos['player_speed_mps'].clip(upper=MAX_PLAYER_SPEED)

print(f"\nValores nulos después de recalcular speed:")
print(f"player_speed_mps: {df_base_videos['player_speed_mps'].isna().sum()}")
print(f"\nEstadísticas de player_speed_mps (después de límite):")
display(df_base_videos['player_speed_mps'].describe())


PASO 4: RECALCULAR VELOCIDAD DEL JUGADOR CON LÍMITES FÍSICOS

Valores que exceden 4.17 m/s: 10681 (3.27%)

Valores nulos después de recalcular speed:
player_speed_mps: 0

Estadísticas de player_speed_mps (después de límite):


count    327066.000000
mean          0.710951
std           1.298654
min           0.000000
25%           0.000000
50%           0.000000
75%           1.697056
max           4.170000
Name: player_speed_mps, dtype: float64

## Paso 6: Calcular aceleración del jugador con límites físicos

Calculamos la aceleración del jugador y aplicamos límites físicos de ±4.7 m/s²

In [ ]:
# Paso 6: Recalcular prev_speed y aceleración del jugador con límites físicos
print("\n" + "="*60)
print("PASO 5: RECALCULAR PREV_SPEED Y ACELERACIÓN CON LÍMITES")
print("="*60)

# Calcular la velocidad previa usando shift por grupo
df_base_videos['prev_speed'] = df_base_videos.groupby(['partido_num', 'player_name_normalized', 'punto'])['player_speed_mps'].shift(1)

# Rellenar los primeros valores de cada grupo con 0
df_base_videos['prev_speed'] = df_base_videos['prev_speed'].fillna(0)

# Calcular aceleración: (speed - prev_speed) / duration
df_base_videos['player_acceleration_mps2'] = (df_base_videos['player_speed_mps'] - df_base_videos['prev_speed']) / df_base_videos['duration']

# Aplicar límites físicos:
# Aceleración máxima del jugador: ~4.5-4.7 m/s²
MAX_PLAYER_ACCELERATION = 4.7  # m/s²
MIN_PLAYER_ACCELERATION = -4.7  # m/s² (desaceleración)

# Identificar valores que exceden el límite
valores_exceden_acc = ((df_base_videos['player_acceleration_mps2'] > MAX_PLAYER_ACCELERATION) | 
                       (df_base_videos['player_acceleration_mps2'] < MIN_PLAYER_ACCELERATION)).sum()
print(f"\nValores que exceden ±{MAX_PLAYER_ACCELERATION} m/s²: {valores_exceden_acc} ({valores_exceden_acc / len(df_base_videos) * 100:.2f}%)")

# Limitar aceleraciones al máximo físico
df_base_videos['player_acceleration_mps2'] = df_base_videos['player_acceleration_mps2'].clip(
    lower=MIN_PLAYER_ACCELERATION, 
    upper=MAX_PLAYER_ACCELERATION
)

print(f"\nValores nulos después de recalcular prev_speed y aceleración:")
print(f"prev_speed: {df_base_videos['prev_speed'].isna().sum()}")
print(f"player_acceleration_mps2: {df_base_videos['player_acceleration_mps2'].isna().sum()}")
print(f"\nEstadísticas de player_acceleration_mps2 (después de límite):")
display(df_base_videos['player_acceleration_mps2'].describe())


PASO 5: RECALCULAR PREV_SPEED Y ACELERACIÓN CON LÍMITES

Valores que exceden ±4.7 m/s²: 108089 (33.05%)

Valores nulos después de recalcular prev_speed y aceleración:
prev_speed: 0
player_acceleration_mps2: 0

Estadísticas de player_acceleration_mps2 (después de límite):


count    327066.000000
mean         -0.004412
std           2.701911
min          -4.700000
25%           0.000000
50%           0.000000
75%           0.000000
max           4.700000
Name: player_acceleration_mps2, dtype: float64

## Resumen final: Variables de movimiento del jugador

Resumen de todas las transformaciones y estadísticas finales de las variables de movimiento

In [ ]:
# Resumen final de la imputación de variables del jugador
print("\n" + "="*60)
print("RESUMEN FINAL - VARIABLES DE DESPLAZAMIENTO DEL JUGADOR")
print("="*60)

print("\nValores nulos finales:")
print(f"player_position_x: {df_base_videos['player_position_x'].isna().sum()}")
print(f"player_position_y: {df_base_videos['player_position_y'].isna().sum()}")
print(f"prev_x: {df_base_videos['prev_x'].isna().sum()}")
print(f"prev_y: {df_base_videos['prev_y'].isna().sum()}")
print(f"player_displacement: {df_base_videos['player_displacement'].isna().sum()}")
print(f"player_speed_mps: {df_base_videos['player_speed_mps'].isna().sum()}")
print(f"prev_speed: {df_base_videos['prev_speed'].isna().sum()}")
print(f"player_acceleration_mps2: {df_base_videos['player_acceleration_mps2'].isna().sum()}")

print("\n\nEstadísticas finales de player_speed_mps:")
display(df_base_videos['player_speed_mps'].describe())

print(f"\n\nRango de velocidades:")
print(f"Mínima: {df_base_videos['player_speed_mps'].min():.2f} m/s ({df_base_videos['player_speed_mps'].min() * 3.6:.2f} km/h)")
print(f"Máxima: {df_base_videos['player_speed_mps'].max():.2f} m/s ({df_base_videos['player_speed_mps'].max() * 3.6:.2f} km/h)")
print(f"Media: {df_base_videos['player_speed_mps'].mean():.2f} m/s ({df_base_videos['player_speed_mps'].mean() * 3.6:.2f} km/h)")
print(f"Mediana: {df_base_videos['player_speed_mps'].median():.2f} m/s ({df_base_videos['player_speed_mps'].median() * 3.6:.2f} km/h)")

print("\n\nEstadísticas finales de player_acceleration_mps2:")
display(df_base_videos['player_acceleration_mps2'].describe())

print(f"\n\nRango de aceleraciones:")
print(f"Mínima: {df_base_videos['player_acceleration_mps2'].min():.2f} m/s²")
print(f"Máxima: {df_base_videos['player_acceleration_mps2'].max():.2f} m/s²")
print(f"Media: {df_base_videos['player_acceleration_mps2'].mean():.2f} m/s²")
print(f"Mediana: {df_base_videos['player_acceleration_mps2'].median():.2f} m/s²")


RESUMEN FINAL - VARIABLES DE DESPLAZAMIENTO DEL JUGADOR

Valores nulos finales:
player_position_x: 0
player_position_y: 0
prev_x: 0
prev_y: 0
player_displacement: 0
player_speed_mps: 0
prev_speed: 0
player_acceleration_mps2: 0


Estadísticas finales de player_speed_mps:


count    327066.000000
mean          0.710951
std           1.298654
min           0.000000
25%           0.000000
50%           0.000000
75%           1.697056
max           4.170000
Name: player_speed_mps, dtype: float64



Rango de velocidades:
Mínima: 0.00 m/s (0.00 km/h)
Máxima: 4.17 m/s (15.01 km/h)
Media: 0.71 m/s (2.56 km/h)
Mediana: 0.00 m/s (0.00 km/h)


Estadísticas finales de player_acceleration_mps2:


count    327066.000000
mean         -0.004412
std           2.701911
min          -4.700000
25%           0.000000
50%           0.000000
75%           0.000000
max           4.700000
Name: player_acceleration_mps2, dtype: float64



Rango de aceleraciones:
Mínima: -4.70 m/s²
Máxima: 4.70 m/s²
Media: -0.00 m/s²
Mediana: 0.00 m/s²


## Verificación final de valores nulos

Verificamos qué columnas aún tienen valores nulos después de todas las imputaciones

In [ ]:
# Calcular el porcentaje de valores nulos por columna
porcentaje_nulos = (df_base_videos.isnull().sum() / len(df_base_videos) * 100).sort_values(ascending=False)
porcentaje_nulos_df = pd.DataFrame({
    'Columna': porcentaje_nulos.index,
    'Porcentaje_Nulos': porcentaje_nulos.values
})

display(porcentaje_nulos_df[porcentaje_nulos_df['Porcentaje_Nulos'] > 0])

,Columna,Porcentaje_Nulos
0,time_since_last_hit,97.782405
1,posicion_cancha,10.347147


## Imputación de posicion_cancha

Rellenamos los valores nulos de posicion_cancha con 'Desconocida'

In [ ]:
df_base_videos['posicion_cancha'] = df_base_videos['posicion_cancha'].fillna('Desconocida')

## Recalcular time_since_last_hit

Calculamos el tiempo transcurrido desde el último golpe usando la variable player_hits_ball

## Verificación inicial de time_since_last_hit

Revisamos el estado de la variable time_since_last_hit antes del recálculo

In [ ]:
# Verificar valores nulos y estadísticas antes del recalculo
print("="*60)
print("RECÁLCULO DE TIME_SINCE_LAST_HIT")
print("="*60)
print(f"\nValores nulos en time_since_last_hit antes: {df_base_videos['time_since_last_hit'].isna().sum()}")
print(f"Porcentaje: {df_base_videos['time_since_last_hit'].isna().sum() / len(df_base_videos) * 100:.2f}%")

print(f"\nValores únicos en player_hits_ball:")
display(df_base_videos['player_hits_ball'].value_counts())

print(f"\nRegistros donde el jugador golpea la pelota: {(df_base_videos['player_hits_ball'] == 1).sum()}")
print(f"Porcentaje: {(df_base_videos['player_hits_ball'] == 1).sum() / len(df_base_videos) * 100:.2f}%")

RECÁLCULO DE TIME_SINCE_LAST_HIT

Valores nulos en time_since_last_hit antes: 319813
Porcentaje: 97.78%

Valores únicos en player_hits_ball:


player_hits_ball
False    324260
True       2806
Name: count, dtype: int64


Registros donde el jugador golpea la pelota: 2806
Porcentaje: 0.86%


## Cálculo de time_since_last_hit

Calculamos el tiempo transcurrido desde el último golpe de pelota por partido y punto

In [ ]:
# Asegurar que el dataframe está ordenado por partido, punto y frame
df_base_videos = df_base_videos.sort_values(['partido_num', 'punto', 'frame_idx'])

# Crear una columna auxiliar para identificar cuándo hay un golpe
# Acumular el tiempo desde el último golpe por partido y punto
def calcular_time_since_last_hit(group):
    """
    Para cada grupo (partido, punto), calcula el tiempo transcurrido desde el último golpe
    """
    # Inicializar con NaN
    time_since_hit = pd.Series(index=group.index, dtype=float)
    
    # Encontrar los índices donde hay golpes
    hit_indices = group[group['player_hits_ball'] == 1].index
    
    if len(hit_indices) == 0:
        # Si no hay golpes en este punto, llenar con NaN o con el tiempo acumulado total
        time_since_hit[:] = (group['duration'].cumsum()).values
    else:
        # Calcular tiempo acumulado desde el inicio
        cumulative_time = group['duration'].cumsum()
        
        # Para cada frame, encontrar el tiempo desde el último golpe
        last_hit_time = 0
        last_hit_idx = None
        
        for idx in group.index:
            current_time = cumulative_time.loc[idx]
            
            # Si este frame tiene un golpe, resetear el tiempo
            if group.loc[idx, 'player_hits_ball'] == 1:
                time_since_hit.loc[idx] = 0
                last_hit_time = current_time
                last_hit_idx = idx
            else:
                # Calcular tiempo desde el último golpe
                if last_hit_idx is not None:
                    time_since_hit.loc[idx] = current_time - last_hit_time
                else:
                    # Si aún no ha habido golpes, usar el tiempo acumulado
                    time_since_hit.loc[idx] = current_time
    
    return time_since_hit

# Aplicar la función por grupo de partido y punto
print("\nCalculando time_since_last_hit...")
df_base_videos['time_since_last_hit'] = df_base_videos.groupby(['partido_num', 'punto']).apply(
    calcular_time_since_last_hit
).reset_index(level=[0, 1], drop=True)

print("Cálculo completado.")


Calculando time_since_last_hit...
Cálculo completado.


## Verificación de resultados de time_since_last_hit

Verificamos que el cálculo de tiempo desde último golpe funcionó correctamente

In [ ]:
# Verificar resultados después del recálculo
print("\n" + "="*60)
print("RESULTADOS DEL RECÁLCULO")
print("="*60)

print(f"\nValores nulos en time_since_last_hit después: {df_base_videos['time_since_last_hit'].isna().sum()}")
print(f"Porcentaje: {df_base_videos['time_since_last_hit'].isna().sum() / len(df_base_videos) * 100:.2f}%")

print(f"\nEstadísticas de time_since_last_hit:")
display(df_base_videos['time_since_last_hit'].describe())

# Mostrar algunos ejemplos de frames con golpes y sus tiempos
print(f"\n\nEjemplos de frames con golpes (time_since_last_hit = 0):")
display(df_base_videos[df_base_videos['player_hits_ball'] == 1][
    ['partido_num', 'punto', 'frame_idx', 'player_name', 'player_hits_ball', 'time_since_last_hit', 'duration']
].head(10))

print(f"\n\nEjemplos de frames sin golpes (con tiempo acumulado):")
display(df_base_videos[df_base_videos['player_hits_ball'] == 0][
    ['partido_num', 'punto', 'frame_idx', 'player_name', 'player_hits_ball', 'time_since_last_hit', 'duration']
].head(10))


RESULTADOS DEL RECÁLCULO

Valores nulos en time_since_last_hit después: 0
Porcentaje: 0.00%

Estadísticas de time_since_last_hit:


count    327066.000000
mean          8.160215
std          11.548153
min           0.000000
25%           1.333333
50%           4.033333
75%          10.100000
max         100.733333
Name: time_since_last_hit, dtype: float64



Ejemplos de frames con golpes (time_since_last_hit = 0):


,partido_num,punto,frame_idx,player_name,player_hits_ball,time_since_last_hit,duration
49494,1,1.0,451,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
49504,1,1.0,461,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
49516,1,1.0,473,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
49517,1,1.0,474,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
49518,1,1.0,475,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
50579,1,3.0,25,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
50580,1,3.0,26,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
50582,1,3.0,28,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
50583,1,3.0,29,Carlos Felipe Andrade Sáenz,True,0.0,0.033333
50584,1,3.0,30,Carlos Felipe Andrade Sáenz,True,0.0,0.033333




Ejemplos de frames sin golpes (con tiempo acumulado):


,partido_num,punto,frame_idx,player_name,player_hits_ball,time_since_last_hit,duration
49210,1,1.0,0,Carlos Felipe Andrade Sáenz,False,0.033333,0.033333
49707,1,1.0,0,Santiago Navarro Cuy,False,0.066667,0.033333
49211,1,1.0,1,Carlos Felipe Andrade Sáenz,False,0.100000,0.033333
49708,1,1.0,1,Santiago Navarro Cuy,False,0.133333,0.033333
49212,1,1.0,2,Carlos Felipe Andrade Sáenz,False,0.166667,0.033333
49709,1,1.0,2,Santiago Navarro Cuy,False,0.200000,0.033333
49213,1,1.0,3,Carlos Felipe Andrade Sáenz,False,0.233333,0.033333
49710,1,1.0,3,Santiago Navarro Cuy,False,0.266667,0.033333
49214,1,1.0,4,Carlos Felipe Andrade Sáenz,False,0.300000,0.033333
49711,1,1.0,4,Santiago Navarro Cuy,False,0.333333,0.033333


## Agregación de variables por jugador-partido

Creamos un dataset agregado con las variables más importantes a nivel jugador-partido para modelado

## Preparación para agregación: verificación de columnas de zona

Verificamos si existen columnas relacionadas con zona de juego para la agregación

In [ ]:
# Crear variable auxiliar para zona ofensiva/defensiva
# Asumiendo que 'zone' contiene información de ataque/defensa
# Si zone no existe o tiene otro formato, ajustar según sea necesario

print("="*60)
print("AGREGACIÓN POR JUGADOR-PARTIDO")
print("="*60)

# Verificar las columnas disponibles relacionadas con zona
print("\nColumnas relacionadas con zona:")
zone_cols = [col for col in df_base_videos.columns if 'zone' in col.lower()]
display(zone_cols)

# Si existe la columna zone, verificar sus valores
if 'zone' in df_base_videos.columns:
    print("\nValores únicos en zone:")
    display(df_base_videos['zone'].value_counts())
else:
    print("\nNota: La columna 'zone' no está disponible")

AGREGACIÓN POR JUGADOR-PARTIDO

Columnas relacionadas con zona:


['zone']


Valores únicos en zone:


zone
left_box      137265
right_box     135899
back_court     53902
Name: count, dtype: int64

## Creación de variables derivadas para agregación

Creamos variables binarias útiles para el análisis agregado (near_ball, zonas ofensiva/defensiva)

In [ ]:
# Crear variables derivadas necesarias para la agregación

# 1. Variable para frames con distancia a pelota < 2m
df_base_videos['near_ball'] = (df_base_videos['distance_player_to_ball_m'] < 2).astype(int)

# 2. Si existe zone, crear variable binaria para zona ofensiva
if 'zone' in df_base_videos.columns:
    # Ajustar según los valores reales de zone
    df_base_videos['in_offensive_zone'] = df_base_videos['zone'].str.contains('attack|offensive', case=False, na=False).astype(int)
    df_base_videos['in_defensive_zone'] = df_base_videos['zone'].str.contains('defense|defensive', case=False, na=False).astype(int)
else:
    # Si no existe zone, crear valores nulos o estimarlos basados en distancia a la red
    # Zona ofensiva: cerca de la red (< 3m por ejemplo)
    df_base_videos['in_offensive_zone'] = (df_base_videos['distance_player_to_net_m'] < 3).astype(int)
    df_base_videos['in_defensive_zone'] = (df_base_videos['distance_player_to_net_m'] >= 3).astype(int)

print("Variables derivadas creadas exitosamente.")

Variables derivadas creadas exitosamente.


## Agregación de datos por jugador-partido

Creamos el dataset agregado calculando estadísticas por jugador en cada partido (medias, sumas, etc.)

In [ ]:
# Crear dataset agregado por jugador-partido
agg_dict = {
    # 1. Velocidad media del jugador
    'player_speed_mps': ['mean', lambda x: x.quantile(0.90), 'std'],
    
    # 3. Distancia total recorrida
    'player_displacement': ['sum', 'mean', 'std'],
    
    # 4. Distancia media a la pelota (y mínima)
    'distance_player_to_ball_m': ['mean', 'min'],
    
    # 5. Distancia media a la red
    'distance_player_to_net_m': 'mean',
    
    # 6. Distancia media al compañero
    'distance_player_to_teammate_m': 'mean',
    
    # 8. Número de golpes (hits)
    'player_hits_ball': 'sum',
    
    # 9. Tiempo promedio desde último golpe
    'time_since_last_hit': 'mean',
    
    # 11. Aceleración media
    'player_acceleration_mps2': ['mean', 'std'],
    
    # 12. Duración total
    'duration': 'sum',
    
    # Variables derivadas
    'near_ball': 'sum',  # Para calcular % frames cerca de pelota
    'in_offensive_zone': 'sum',  # Para calcular % tiempo en zona ofensiva
    'in_defensive_zone': 'sum'  # Para calcular tiempo en zona defensiva
}

# Realizar la agregación
df_player_partido = df_base_videos.groupby(['player_name_normalized', 'partido_num']).agg(agg_dict)

# Aplanar columnas multi-nivel
df_player_partido.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                              for col in df_player_partido.columns.values]

# Renombrar columnas para mayor claridad
rename_dict = {
    'player_speed_mps_mean': 'velocidad_media_mps',
    'player_speed_mps_<lambda>': 'velocidad_p90_mps',
    'player_speed_mps_std': 'velocidad_std_mps',
    'player_displacement_sum': 'distancia_total_m',
    'player_displacement_mean': 'distancia_media_por_frame_m',
    'player_displacement_std': 'desplazamiento_std_m',
    'distance_player_to_ball_m_mean': 'distancia_media_pelota_m',
    'distance_player_to_ball_m_min': 'distancia_min_pelota_m',
    'distance_player_to_net_m_mean': 'distancia_media_red_m',
    'distance_player_to_teammate_m_mean': 'distancia_media_companero_m',
    'player_hits_ball_sum': 'num_golpes',
    'time_since_last_hit_mean': 'tiempo_promedio_desde_ultimo_golpe_s',
    'player_acceleration_mps2_mean': 'aceleracion_media_mps2',
    'player_acceleration_mps2_std': 'aceleracion_std_mps2',
    'duration_sum': 'duracion_total_s',
    'near_ball_sum': 'frames_cerca_pelota',
    'in_offensive_zone_sum': 'frames_zona_ofensiva',
    'in_defensive_zone_sum': 'frames_zona_defensiva'
}

df_player_partido.rename(columns=rename_dict, inplace=True)

# Calcular variables derivadas adicionales
# Total de frames por jugador-partido
df_player_partido['total_frames'] = df_base_videos.groupby(['player_name_normalized', 'partido_num']).size()

# Porcentaje de frames cerca de la pelota (< 2m)
df_player_partido['pct_frames_cerca_pelota'] = (df_player_partido['frames_cerca_pelota'] / 
                                                  df_player_partido['total_frames'] * 100)

# Porcentaje de tiempo en zona ofensiva
df_player_partido['pct_tiempo_zona_ofensiva'] = (df_player_partido['frames_zona_ofensiva'] / 
                                                   df_player_partido['total_frames'] * 100)

# Tiempo total en zona defensiva (en segundos)
# Asumiendo que cada frame tiene un duration, usamos la proporción de frames
df_player_partido['tiempo_zona_defensiva_s'] = (df_player_partido['frames_zona_defensiva'] / 
                                                  df_player_partido['total_frames']) * df_player_partido['duracion_total_s']

# Resetear índice para tener jugador y partido como columnas
df_player_partido.reset_index(inplace=True)

print(f"Dataset agregado creado: {df_player_partido.shape[0]} registros (jugador-partido)")
print(f"Columnas: {df_player_partido.shape[1]}")
display(df_player_partido.head(10))

Dataset agregado creado: 104 registros (jugador-partido)
Columnas: 24


,player_name_normalized,partido_num,velocidad_media_mps,player_speed_mps_<lambda_0>,velocidad_std_mps,distancia_total_m,distancia_media_por_frame_m,desplazamiento_std_m,distancia_media_pelota_m,distancia_min_pelota_m,...,aceleracion_media_mps2,aceleracion_std_mps2,duracion_total_s,frames_cerca_pelota,frames_zona_ofensiva,frames_zona_defensiva,total_frames,pct_frames_cerca_pelota,pct_tiempo_zona_ofensiva,tiempo_zona_defensiva_s
0,andrea julieth sosa rodriguez,17,0.443454,1.697056,0.953379,161.955737,0.055012,0.398716,5.880587,0.283505,...,-0.012772,2.468738,98.133333,186,0,0,2944,6.317935,0.0,0.0
1,andrea julieth sosa rodriguez,23,0.538182,1.697056,0.999859,141.817336,0.062640,0.400295,11.885809,0.119588,...,-0.006228,2.764619,75.466667,30,0,0,2264,1.325088,0.0,0.0
2,andrea julieth sosa rodriguez,29,0.751067,1.697056,1.179528,401.862926,0.090266,0.744584,7.794564,0.200000,...,-0.010557,2.926757,148.400000,271,0,0,4452,6.087152,0.0,0.0
3,carlos felipe andrade saenz,1,0.526525,1.697056,1.023018,57.247365,0.036862,0.259287,8.597797,1.048512,...,-0.009079,2.555969,51.766667,177,0,0,1553,11.397296,0.0,0.0
4,carlos felipe andrade saenz,19,1.722628,4.170000,1.456344,723.624796,0.228561,1.202169,10.652102,2.211195,...,0.025237,3.742612,105.533333,0,0,0,3166,0.000000,0.0,0.0
5,carlos felipe andrade saenz,21,0.648310,1.697056,1.173588,354.911036,0.112420,0.507808,4.798010,0.040000,...,-0.019354,2.742981,105.233333,586,0,0,3157,18.561926,0.0,0.0
6,cesar camilo diaz cufino,9,0.867422,3.394113,1.542613,384.496383,0.058818,0.418233,5.999708,0.120000,...,-0.006471,2.538726,108.950000,709,0,0,6537,10.845954,0.0,0.0
7,cesar camilo diaz cufino,14,0.353326,3.394113,1.043873,129.089414,0.033253,0.568454,7.058445,0.040000,...,-0.001211,1.939665,64.700000,486,0,0,3882,12.519320,0.0,0.0
8,cristian stevan soto zapata,32,0.801852,1.697056,1.092823,201.949697,0.091339,0.798494,6.445585,0.976322,...,-0.034012,3.282372,73.700000,26,0,0,2211,1.175938,0.0,0.0
9,cristian stevan soto zapata,41,0.320100,1.697056,0.754841,36.656416,0.017547,0.135562,3.574061,0.621968,...,0.002250,2.241709,69.633333,739,0,0,2089,35.375778,0.0,0.0


## Estadísticas descriptivas del dataset agregado

Mostramos un resumen estadístico del dataset agregado por jugador-partido

In [ ]:
# Verificar estadísticas descriptivas del dataset agregado
print("Estadísticas descriptivas del dataset agregado:")
display(df_player_partido.describe())

print("\nPrimeras 5 filas:")
display(df_player_partido.head())

print("\nVerificación de nulos:")
nulls = df_player_partido.isnull().sum()
display(nulls[nulls > 0] if nulls.sum() > 0 else "No hay valores nulos")

Estadísticas descriptivas del dataset agregado:


,partido_num,velocidad_media_mps,player_speed_mps_<lambda_0>,velocidad_std_mps,distancia_total_m,distancia_media_por_frame_m,desplazamiento_std_m,distancia_media_pelota_m,distancia_min_pelota_m,distancia_media_red_m,...,aceleracion_media_mps2,aceleracion_std_mps2,duracion_total_s,frames_cerca_pelota,frames_zona_ofensiva,frames_zona_defensiva,total_frames,pct_frames_cerca_pelota,pct_tiempo_zona_ofensiva,tiempo_zona_defensiva_s
count,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,104.000000,...,104.000000,104.000000,104.000000,104.000000,104.0,104.0,104.000000,104.000000,104.0,104.0
mean,24.884615,0.717287,2.778701,1.243216,200.513182,0.071358,0.541242,6.668900,0.751379,4.764580,...,-0.004611,2.687564,76.876122,313.894231,0.0,0.0,3144.865385,9.986258,0.0,0.0
std,12.587795,0.297824,0.906192,0.279146,126.802533,0.045537,0.247054,2.373831,1.082761,1.780990,...,0.011101,0.385722,39.821622,413.845263,0.0,0.0,1308.234503,12.720358,0.0,0.0
min,1.000000,0.204742,0.000000,0.597759,36.656416,0.011415,0.096482,1.399087,0.000000,0.717631,...,-0.039698,1.837163,15.450000,0.000000,0.0,0.0,927.000000,0.000000,0.0,0.0
25%,14.000000,0.490527,1.697056,0.995823,116.983746,0.035685,0.378769,5.116533,0.139204,3.327293,...,-0.009871,2.468107,53.333333,26.000000,0.0,0.0,2096.500000,1.128639,0.0,0.0
50%,27.000000,0.654197,3.394113,1.253491,170.921851,0.061108,0.553740,6.135509,0.340756,4.627598,...,-0.003066,2.659377,74.383333,161.000000,0.0,0.0,3161.500000,6.246336,0.0,0.0
75%,36.000000,0.949312,3.394113,1.454923,246.158013,0.094994,0.708519,7.964268,0.877652,5.899331,...,0.002243,2.925875,99.291667,451.000000,0.0,0.0,3988.000000,13.382087,0.0,0.0
max,44.000000,1.722628,4.170000,1.777129,723.624796,0.269796,1.202169,14.602281,7.295439,8.489038,...,0.025237,3.742612,173.933333,2107.000000,0.0,0.0,6537.000000,86.814998,0.0,0.0



Primeras 5 filas:


,player_name_normalized,partido_num,velocidad_media_mps,player_speed_mps_<lambda_0>,velocidad_std_mps,distancia_total_m,distancia_media_por_frame_m,desplazamiento_std_m,distancia_media_pelota_m,distancia_min_pelota_m,...,aceleracion_media_mps2,aceleracion_std_mps2,duracion_total_s,frames_cerca_pelota,frames_zona_ofensiva,frames_zona_defensiva,total_frames,pct_frames_cerca_pelota,pct_tiempo_zona_ofensiva,tiempo_zona_defensiva_s
0,andrea julieth sosa rodriguez,17,0.443454,1.697056,0.953379,161.955737,0.055012,0.398716,5.880587,0.283505,...,-0.012772,2.468738,98.133333,186,0,0,2944,6.317935,0.0,0.0
1,andrea julieth sosa rodriguez,23,0.538182,1.697056,0.999859,141.817336,0.062640,0.400295,11.885809,0.119588,...,-0.006228,2.764619,75.466667,30,0,0,2264,1.325088,0.0,0.0
2,andrea julieth sosa rodriguez,29,0.751067,1.697056,1.179528,401.862926,0.090266,0.744584,7.794564,0.200000,...,-0.010557,2.926757,148.400000,271,0,0,4452,6.087152,0.0,0.0
3,carlos felipe andrade saenz,1,0.526525,1.697056,1.023018,57.247365,0.036862,0.259287,8.597797,1.048512,...,-0.009079,2.555969,51.766667,177,0,0,1553,11.397296,0.0,0.0
4,carlos felipe andrade saenz,19,1.722628,4.170000,1.456344,723.624796,0.228561,1.202169,10.652102,2.211195,...,0.025237,3.742612,105.533333,0,0,0,3166,0.000000,0.0,0.0



Verificación de nulos:


'No hay valores nulos'

## Guardar dataset agregado

Guardamos el dataset agregado por jugador-partido en un archivo CSV

In [ ]:
# Guardar el dataset agregado
output_file = DATA_DIR / 'interim' / 'Jugador_Partido_Features.csv'
df_player_partido.to_csv(output_file, index=False, encoding='utf-8')
print(f"Dataset agregado guardado en: {output_file}")
print(f"Dimensiones: {df_player_partido.shape}")
print(f"\nColumnas incluidas ({len(df_player_partido.columns)}):")
for i, col in enumerate(df_player_partido.columns, 1):
    print(f"  {i}. {col}")

Dataset agregado guardado en: Jugador_Partido_Features.csv
Dimensiones: (104, 24)

Columnas incluidas (24):
  1. player_name_normalized
  2. partido_num
  3. velocidad_media_mps
  4. player_speed_mps_<lambda_0>
  5. velocidad_std_mps
  6. distancia_total_m
  7. distancia_media_por_frame_m
  8. desplazamiento_std_m
  9. distancia_media_pelota_m
  10. distancia_min_pelota_m
  11. distancia_media_red_m
  12. distancia_media_companero_m
  13. num_golpes
  14. tiempo_promedio_desde_ultimo_golpe_s
  15. aceleracion_media_mps2
  16. aceleracion_std_mps2
  17. duracion_total_s
  18. frames_cerca_pelota
  19. frames_zona_ofensiva
  20. frames_zona_defensiva
  21. total_frames
  22. pct_frames_cerca_pelota
  23. pct_tiempo_zona_ofensiva
  24. tiempo_zona_defensiva_s


## Resultados de partidos

## Generación de dataset de resultados por partido

Creamos un dataset con los resultados finales de cada partido sumando los puntos de cada equipo

In [ ]:
# Generar dataset de resultados de partidos
# PUNTO_EQUIPO_1 y PUNTO_EQUIPO_2 indican quién ganó cada punto (1 o 0)
# Necesitamos sumar todos los puntos para obtener el resultado final

# Extraer número de partido de ID_PARTIDO si no existe
if 'partido_num' not in df_levantamiento.columns:
    df_levantamiento['partido_num'] = df_levantamiento['ID_PARTIDO'].apply(extraer_numero_partido)

# Convertir columnas de puntos a numéricas
df_levantamiento['PUNTO_EQUIPO_1'] = pd.to_numeric(df_levantamiento['PUNTO_EQUIPO_1'], errors='coerce')
df_levantamiento['PUNTO_EQUIPO_2'] = pd.to_numeric(df_levantamiento['PUNTO_EQUIPO_2'], errors='coerce')

# Agrupar por partido y SUMAR los puntos de cada equipo (no máximo)
resultados_partidos = df_levantamiento.groupby('partido_num').agg({
    'PUNTO_EQUIPO_1': 'sum',
    'PUNTO_EQUIPO_2': 'sum'
}).reset_index()

# Renombrar columnas
resultados_partidos.rename(columns={
    'PUNTO_EQUIPO_1': 'puntos_team_1',
    'PUNTO_EQUIPO_2': 'puntos_team_2'
}, inplace=True)

# Determinar ganador del partido
resultados_partidos['ganador'] = resultados_partidos.apply(
    lambda row: 'team_1' if row['puntos_team_1'] > row['puntos_team_2'] else 'team_2', 
    axis=1
)

# Convertir a enteros
resultados_partidos['puntos_team_1'] = resultados_partidos['puntos_team_1'].astype(int)
resultados_partidos['puntos_team_2'] = resultados_partidos['puntos_team_2'].astype(int)

display(resultados_partidos.head(10))
print(f"\nTotal de partidos: {len(resultados_partidos)}")
print(f"Partidos ganados por team_1: {(resultados_partidos['ganador'] == 'team_1').sum()}")
print(f"Partidos ganados por team_2: {(resultados_partidos['ganador'] == 'team_2').sum()}")

,partido_num,puntos_team_1,puntos_team_2,ganador
0,1,7,1,team_1
1,2,7,2,team_1
2,3,5,7,team_2
3,4,7,2,team_1
4,5,7,6,team_1
5,6,6,6,team_2
6,7,2,7,team_2
7,8,7,4,team_1
8,9,6,5,team_1
9,10,6,6,team_2



Total de partidos: 43
Partidos ganados por team_1: 19
Partidos ganados por team_2: 24


## Verificación de estructura de puntos

Verificamos cómo están estructurados los puntos en el dataset original

In [ ]:
# Verificar datos originales para entender la estructura de puntos
print("Verificando estructura de puntos en el dataset original:")
display(df_levantamiento[['partido_num', 'ID_PUNTO', 'PUNTO_EQUIPO_1', 'PUNTO_EQUIPO_2']].head(20))

# Ver algunos partidos completos
print("\nEjemplo de Partido 1 completo:")
partido_1 = df_levantamiento[df_levantamiento['partido_num'] == 1][['partido_num', 'ID_PUNTO', 'PUNTO_EQUIPO_1', 'PUNTO_EQUIPO_2']]
display(partido_1)

Verificando estructura de puntos en el dataset original:


,partido_num,ID_PUNTO,PUNTO_EQUIPO_1,PUNTO_EQUIPO_2
0,27,1,1.0,0.0
1,27,2,0.0,1.0
2,27,3,0.0,1.0
3,27,4,1.0,0.0
4,27,5,0.0,1.0
5,27,6,1.0,0.0
6,27,7,0.0,1.0
7,27,8,1.0,0.0
8,27,9,0.0,1.0
9,27,10,1.0,0.0



Ejemplo de Partido 1 completo:


,partido_num,ID_PUNTO,PUNTO_EQUIPO_1,PUNTO_EQUIPO_2
202,1,1,1.0,0.0
203,1,2,0.0,1.0
204,1,3,1.0,0.0
205,1,4,1.0,0.0
206,1,5,1.0,0.0
207,1,6,1.0,0.0
208,1,7,1.0,0.0
209,1,8,1.0,0.0


## Guardar dataset de resultados

Guardamos el dataset de resultados de partidos en un archivo CSV

In [ ]:
# Guardar el dataset de resultados
resultados_partidos.to_csv(DATA_DIR / 'processed' / 'Resultados_Partidos.csv', index=False, encoding='utf-8')
print(f"Dataset de resultados guardado: Resultados_Partidos.csv")
print(f"Dimensiones: {resultados_partidos.shape}")

Dataset de resultados guardado: Resultados_Partidos.csv
Dimensiones: (43, 4)


## Añadir información de victoria al dataset de jugador-partido

Integramos los resultados de partidos con el dataset agregado para saber si cada jugador ganó o perdió

In [ ]:
# Añadir información de si ganó o no el partido al dataset de jugador-partido
# Primero necesitamos mapear cada jugador a su equipo en cada partido

# Crear un mapeo de jugador-partido a equipo desde df_base_videos
jugador_partido_equipo = df_base_videos[['player_name_normalized', 'partido_num', 'equipo']].drop_duplicates()

# Mergear con el dataset agregado
df_player_partido = df_player_partido.merge(
    jugador_partido_equipo,
    on=['player_name_normalized', 'partido_num'],
    how='left'
)

# Extraer el número de equipo del formato 'match_X_team_Y'
df_player_partido['team_num'] = df_player_partido['equipo'].str.extract(r'team_(\d+)').astype(int)

# Mergear con resultados de partidos
df_player_partido = df_player_partido.merge(
    resultados_partidos,
    on='partido_num',
    how='left'
)

# Determinar si ganó el partido
df_player_partido['gano_partido'] = df_player_partido.apply(
    lambda row: 1 if f"team_{row['team_num']}" == row['ganador'] else 0,
    axis=1
)

# Verificar el resultado
print("Columnas añadidas: equipo, team_num, puntos_team_1, puntos_team_2, ganador, gano_partido")
print(f"\nJugadores que ganaron: {df_player_partido['gano_partido'].sum()}")
print(f"Jugadores que perdieron: {(df_player_partido['gano_partido'] == 0).sum()}")
display(df_player_partido[['player_name_normalized', 'partido_num', 'equipo', 'puntos_team_1', 'puntos_team_2', 'ganador', 'gano_partido']].head(10))

Columnas añadidas: equipo, team_num, puntos_team_1, puntos_team_2, ganador, gano_partido

Jugadores que ganaron: 52
Jugadores que perdieron: 52


,player_name_normalized,partido_num,equipo,puntos_team_1,puntos_team_2,ganador,gano_partido
0,andrea julieth sosa rodriguez,17,match_17_team_2,6,6,team_2,1
1,andrea julieth sosa rodriguez,23,match_23_team_2,7,5,team_1,0
2,andrea julieth sosa rodriguez,29,match_29_team_1,7,5,team_1,1
3,carlos felipe andrade saenz,1,match_1_team_2,7,1,team_1,0
4,carlos felipe andrade saenz,19,match_19_team_2,7,4,team_1,0
5,carlos felipe andrade saenz,21,match_21_team_2,2,7,team_2,1
6,cesar camilo diaz cufino,9,match_9_team_2,6,5,team_1,0
7,cesar camilo diaz cufino,14,match_14_team_2,7,4,team_1,0
8,cristian stevan soto zapata,32,match_32_team_1,7,5,team_1,1
9,cristian stevan soto zapata,41,match_41_team_2,5,7,team_2,1


## Guardar dataset actualizado con victoria

Guardamos el dataset de jugador-partido con la columna de victoria añadida

In [ ]:
# Guardar el dataset actualizado con la información de victoria
df_player_partido.to_csv(DATA_DIR / 'interim' / 'Jugador_Partido_Features.csv', index=False, encoding='utf-8')
print(f"Dataset actualizado guardado: Jugador_Partido_Features.csv")
print(f"Dimensiones: {df_player_partido.shape}")
print(f"\nColumnas finales ({len(df_player_partido.columns)}):")
for i, col in enumerate(df_player_partido.columns, 1):
    print(f"  {i}. {col}")

Dataset actualizado guardado: Jugador_Partido_Features.csv
Dimensiones: (104, 30)

Columnas finales (30):
  1. player_name_normalized
  2. partido_num
  3. velocidad_media_mps
  4. player_speed_mps_<lambda_0>
  5. velocidad_std_mps
  6. distancia_total_m
  7. distancia_media_por_frame_m
  8. desplazamiento_std_m
  9. distancia_media_pelota_m
  10. distancia_min_pelota_m
  11. distancia_media_red_m
  12. distancia_media_companero_m
  13. num_golpes
  14. tiempo_promedio_desde_ultimo_golpe_s
  15. aceleracion_media_mps2
  16. aceleracion_std_mps2
  17. duracion_total_s
  18. frames_cerca_pelota
  19. frames_zona_ofensiva
  20. frames_zona_defensiva
  21. total_frames
  22. pct_frames_cerca_pelota
  23. pct_tiempo_zona_ofensiva
  24. tiempo_zona_defensiva_s
  25. equipo
  26. team_num
  27. puntos_team_1
  28. puntos_team_2
  29. ganador
  30. gano_partido


## Integración con datos de jugadores

## Cargar datos de perfil de jugadores

Cargamos el archivo con información personal y características de los jugadores (edad, experiencia, etc.)

In [ ]:
# Cargar datos de jugadores
df_jugadores = pd.read_csv(DATA_DIR / 'raw' / 'Datos_Jugadores_Padel(Respuestas de formulario 1).csv', encoding='latin1')

print(f"Datos de jugadores cargados: {df_jugadores.shape}")
print(f"\nColumnas disponibles:")
display(df_jugadores.columns.tolist())
print(f"\nPrimeras filas:")
display(df_jugadores.head())

Datos de jugadores cargados: (35, 27)

Columnas disponibles:


['Marca temporal',
 'NOMBRE',
 'CODIGO_ESTUDIANTE',
 'CIUDAD',
 'EDAD',
 'GENERO',
 'PROGRAMA_ACADEMICO',
 'SEMESTRE',
 'EXPERIENCIA_PADEL',
 'TIEMPO_JUGANDO_PADEL',
 'PRACTICA_OTRO_DEPORTE_RAQUETA',
 'NIVEL_ACTUAL_PADEL',
 'ESTADO_FISICO',
 'FRECUENCIA_DEPORTE',
 'ESTATURA',
 'TALLA',
 'MANO_DOMINANTE',
 'RITMO_CARDIACO',
 'LESIONES_LY',
 'RAZON_PARTICIPACION',
 'OBJETIVO_ACTIVIDAD',
 'INTERES_PADEL',
 'CLASES_PADEL',
 'DESAYUNO',
 '¿Que desayunaste?',
 'GENERO_MUSICAL_FAVORITO',
 'COMENTARIOS']


Primeras filas:


,Marca temporal,NOMBRE,CODIGO_ESTUDIANTE,CIUDAD,EDAD,GENERO,PROGRAMA_ACADEMICO,SEMESTRE,EXPERIENCIA_PADEL,TIEMPO_JUGANDO_PADEL,...,RITMO_CARDIACO,LESIONES_LY,RAZON_PARTICIPACION,OBJETIVO_ACTIVIDAD,INTERES_PADEL,CLASES_PADEL,DESAYUNO,¿Que desayunaste?,GENERO_MUSICAL_FAVORITO,COMENTARIOS
0,8/7/2025 8:40:19,Cristian Stevan Soto Zapata,260647,Sopo,22,Masculino,Ingeniería Informática,9,No,Menos de 6 meses,...,NaN,No,Por la materia,Aprender,Sí,Sí,Si,Huevos,Reggaeton,Excelente actividad
1,8/7/2025 8:40:30,Maria Fernanda Tamayo Lopez,242491,Cajica,21,Femenino,Ingeniería Informática,7,No,Menos de 6 meses,...,67,No,Me gustan los deportes,Aprender a jugar Padel,Tal vez,Sí,Si,"Calentado, huevo frito y salchicha",Rock,Pan con queso
2,8/7/2025 8:40:56,Tomas Barrios Guevara,299043,Bogota - Occidente,20,Masculino,Ingeniería Informática,6,No,Menos de 6 meses,...,90,Sí,Proyecto y aprender,Identificar como juego y patrones de las personas,Tal vez,Sí,Si,Pastel de carne,Reggaeton,Que sea una buena experiencia
3,8/7/2025 8:40:57,Santiago Navarro Cuy,305992,Cajica,19,Masculino,Ingeniería Informática,6,No,Menos de 6 meses,...,NaN,No,Probar un deporte nuevo,Jugar un nuevo deporte,Tal vez,Tal vez,Si,tamal,Electronica,Buena actividad
4,8/7/2025 8:40:57,Fermín Alejandro Escalona Guillen,300181,Bogota - Norte,19,Masculino,Ingeniería Informática,6,No,Menos de 6 meses,...,NaN,Sí,Diversión,Divertirme,Sí,Sí,Si,Empanada con cafe,Reggaeton,Qué actividad tan divertida


## Preparación de datos de jugadores para merge

Normalizamos nombres y seleccionamos columnas relevantes del perfil de jugadores

In [ ]:
# Normalizar nombres en el dataset de jugadores
df_jugadores['nombre_normalizado'] = df_jugadores['NOMBRE'].apply(normalizar_nombre)

# Seleccionar columnas relevantes para el merge
columnas_jugador = [
    'nombre_normalizado', 'EDAD', 'GENERO', 'EXPERIENCIA_PADEL', 
    'TIEMPO_JUGANDO_PADEL', 'NIVEL_ACTUAL_PADEL', 'ESTADO_FISICO',
    'FRECUENCIA_DEPORTE', 'ESTATURA', 'TALLA', 'MANO_DOMINANTE',
    'PRACTICA_OTRO_DEPORTE_RAQUETA'
]

df_jugadores_info = df_jugadores[columnas_jugador].copy()

print(f"Jugadores con información: {len(df_jugadores_info)}")
print(f"\nColumnas a añadir:")
display(columnas_jugador[1:])  # Excluir nombre_normalizado
print(f"\nPrimeras filas procesadas:")
display(df_jugadores_info.head())

Jugadores con información: 35

Columnas a añadir:


['EDAD',
 'GENERO',
 'EXPERIENCIA_PADEL',
 'TIEMPO_JUGANDO_PADEL',
 'NIVEL_ACTUAL_PADEL',
 'ESTADO_FISICO',
 'FRECUENCIA_DEPORTE',
 'ESTATURA',
 'TALLA',
 'MANO_DOMINANTE',
 'PRACTICA_OTRO_DEPORTE_RAQUETA']


Primeras filas procesadas:


,nombre_normalizado,EDAD,GENERO,EXPERIENCIA_PADEL,TIEMPO_JUGANDO_PADEL,NIVEL_ACTUAL_PADEL,ESTADO_FISICO,FRECUENCIA_DEPORTE,ESTATURA,TALLA,MANO_DOMINANTE,PRACTICA_OTRO_DEPORTE_RAQUETA
0,cristian stevan soto zapata,22,Masculino,No,Menos de 6 meses,Primera vez,Bueno,3-4 veces,172,68.0,Derecha,No
1,maria fernanda tamayo lopez,21,Femenino,No,Menos de 6 meses,Primera vez,Bueno,3-4 veces,167,55.0,Derecha,Tenis de mesa
2,tomas barrios guevara,20,Masculino,No,Menos de 6 meses,Principiante,Bueno,3-4 veces,168,70.0,Derecha,No
3,santiago navarro cuy,19,Masculino,No,Menos de 6 meses,Primera vez,Malo,0 veces,180,85.0,Derecha,No
4,fermin alejandro escalona guillen,19,Masculino,No,Menos de 6 meses,Primera vez,Excelente,Más de 4 veces,176,75.0,Derecha,No


## Integrar datos de perfil de jugadores

Hacemos merge del dataset agregado con la información personal de los jugadores

In [ ]:
# Mergear con el dataset de jugador-partido
df_player_partido_completo = df_player_partido.merge(
    df_jugadores_info,
    left_on='player_name_normalized',
    right_on='nombre_normalizado',
    how='left'
)

# Eliminar columna duplicada
df_player_partido_completo.drop('nombre_normalizado', axis=1, inplace=True)

print(f"Dataset después del merge: {df_player_partido_completo.shape}")
print(f"\nJugadores sin información de perfil:")
jugadores_desconocido = df_player_partido_completo[df_player_partido_completo['EDAD'].isna()]['player_name_normalized'].unique()
print(f"Total: {len(jugadores_desconocido)}")
display(jugadores_desconocido)

print(f"\nNulos por columna después del merge:")
nulos_restantes = df_player_partido_completo.isnull().sum()
display(nulos_restantes[nulos_restantes > 0])

Dataset después del merge: (107, 41)

Jugadores sin información de perfil:
Total: 9


array(['carlos felipe andrade saenz', 'jaime andres castillo alvarez',
       'jorge enrique lugo lopez', 'luis jorge garcia camargo',
       'maria jose cifuentes', 'nicolas navarro', 'nicole lizarazo',
       'nikol sofia avendano', 'santiago gutierrez de pineres barbosa'],
      dtype=object)


Nulos por columna después del merge:


EDAD                             31
GENERO                           31
EXPERIENCIA_PADEL                31
TIEMPO_JUGANDO_PADEL             31
NIVEL_ACTUAL_PADEL               31
ESTADO_FISICO                    31
FRECUENCIA_DEPORTE               31
ESTATURA                         31
TALLA                            31
MANO_DOMINANTE                   31
PRACTICA_OTRO_DEPORTE_RAQUETA    31
dtype: int64

## Imputación de valores faltantes de perfil

Imputamos valores faltantes: categóricas con 'Desconocido', numéricas con mediana

In [ ]:
# Imputar valores faltantes
# Columnas categóricas: llenar con "Desconocido"
columnas_categoricas = ['GENERO', 'EXPERIENCIA_PADEL', 'TIEMPO_JUGANDO_PADEL', 
                        'NIVEL_ACTUAL_PADEL', 'ESTADO_FISICO', 'FRECUENCIA_DEPORTE',
                        'MANO_DOMINANTE', 'PRACTICA_OTRO_DEPORTE_RAQUETA']

for col in columnas_categoricas:
    df_player_partido_completo[col].fillna('Desconocido', inplace=True)

# Columnas numéricas: imputar con la mediana
columnas_numericas = ['EDAD', 'ESTATURA', 'TALLA']

for col in columnas_numericas:
    mediana = df_player_partido_completo[col].median()
    df_player_partido_completo[col].fillna(mediana, inplace=True)
    print(f"{col}: imputado con mediana = {mediana}")

print(f"\nVerificación de nulos después de la imputación:")
nulos_finales = df_player_partido_completo.isnull().sum()
display(nulos_finales[nulos_finales > 0] if nulos_finales.sum() > 0 else "No hay valores nulos")

print(f"\nDataset completo: {df_player_partido_completo.shape}")
display(df_player_partido_completo.head())

EDAD: imputado con mediana = 19.0
ESTATURA: imputado con mediana = 173.0
TALLA: imputado con mediana = 64.0

Verificación de nulos después de la imputación:


'No hay valores nulos'


Dataset completo: (107, 41)


,player_name_normalized,partido_num,velocidad_media_mps,player_speed_mps_<lambda_0>,velocidad_std_mps,distancia_total_m,distancia_media_por_frame_m,desplazamiento_std_m,distancia_media_pelota_m,distancia_min_pelota_m,...,GENERO,EXPERIENCIA_PADEL,TIEMPO_JUGANDO_PADEL,NIVEL_ACTUAL_PADEL,ESTADO_FISICO,FRECUENCIA_DEPORTE,ESTATURA,TALLA,MANO_DOMINANTE,PRACTICA_OTRO_DEPORTE_RAQUETA
0,andrea julieth sosa rodriguez,17,0.443454,1.697056,0.953379,161.955737,0.055012,0.398716,5.880587,0.283505,...,Femenino,No,Menos de 6 meses,Primera vez,Bueno,3-4 veces,162.0,50.0,Derecha,Tenis
1,andrea julieth sosa rodriguez,23,0.538182,1.697056,0.999859,141.817336,0.062640,0.400295,11.885809,0.119588,...,Femenino,No,Menos de 6 meses,Primera vez,Bueno,3-4 veces,162.0,50.0,Derecha,Tenis
2,andrea julieth sosa rodriguez,29,0.751067,1.697056,1.179528,401.862926,0.090266,0.744584,7.794564,0.200000,...,Femenino,No,Menos de 6 meses,Primera vez,Bueno,3-4 veces,162.0,50.0,Derecha,Tenis
3,carlos felipe andrade saenz,1,0.526525,1.697056,1.023018,57.247365,0.036862,0.259287,8.597797,1.048512,...,Desconocido,Desconocido,Desconocido,Desconocido,Desconocido,Desconocido,173.0,64.0,Desconocido,Desconocido
4,carlos felipe andrade saenz,19,1.722628,4.170000,1.456344,723.624796,0.228561,1.202169,10.652102,2.211195,...,Desconocido,Desconocido,Desconocido,Desconocido,Desconocido,Desconocido,173.0,64.0,Desconocido,Desconocido


## Guardar dataset completo final

Guardamos el dataset completo con todas las variables: rendimiento, perfil y resultados

In [ ]:
# Guardar el dataset completo
df_player_partido_completo.to_csv(DATA_DIR / 'processed' / 'Jugador_Partido_Features_Completo.csv', index=False, encoding='utf-8')
print(f"Dataset completo guardado: Jugador_Partido_Features_Completo.csv")
print(f"Dimensiones finales: {df_player_partido_completo.shape}")
print(f"\nColumnas totales ({len(df_player_partido_completo.columns)}):")
for i, col in enumerate(df_player_partido_completo.columns, 1):
    print(f"  {i}. {col}")

Dataset completo guardado: Jugador_Partido_Features_Completo.csv
Dimensiones finales: (107, 41)

Columnas totales (41):
  1. player_name_normalized
  2. partido_num
  3. velocidad_media_mps
  4. player_speed_mps_<lambda_0>
  5. velocidad_std_mps
  6. distancia_total_m
  7. distancia_media_por_frame_m
  8. desplazamiento_std_m
  9. distancia_media_pelota_m
  10. distancia_min_pelota_m
  11. distancia_media_red_m
  12. distancia_media_companero_m
  13. num_golpes
  14. tiempo_promedio_desde_ultimo_golpe_s
  15. aceleracion_media_mps2
  16. aceleracion_std_mps2
  17. duracion_total_s
  18. frames_cerca_pelota
  19. frames_zona_ofensiva
  20. frames_zona_defensiva
  21. total_frames
  22. pct_frames_cerca_pelota
  23. pct_tiempo_zona_ofensiva
  24. tiempo_zona_defensiva_s
  25. equipo
  26. team_num
  27. puntos_team_1
  28. puntos_team_2
  29. ganador
  30. gano_partido
  31. EDAD
  32. GENERO
  33. EXPERIENCIA_PADEL
  34. TIEMPO_JUGANDO_PADEL
  35. NIVEL_ACTUAL_PADEL
  36. ESTADO_FISICO



# 👥 DATASET DE EQUIPOS

Creación de dataset agregado por equipo-partido

In [ ]:
# Crear dataset de equipos agregando métricas de ambos jugadores
print("="*80)
print("CREACIÓN DE DATASET DE EQUIPOS")
print("="*80)

# Agrupar por partido y equipo
df_equipos = df_player_partido_completo.groupby(['partido_num', 'equipo']).agg({
    # Identificadores
    'team_num': 'first',
    'ganador': 'first',
    'puntos_team_1': 'first',
    'puntos_team_2': 'first',
    
    # Métricas agregadas del equipo (promedio de ambos jugadores)
    'velocidad_media_mps': 'mean',
    'player_speed_mps_<lambda_0>': 'mean',
    'velocidad_std_mps': 'mean',
    'distancia_total_m': 'sum',  # Suma porque es la distancia total del equipo
    'distancia_media_por_frame_m': 'mean',
    'desplazamiento_std_m': 'mean',
    'distancia_media_pelota_m': 'mean',
    'distancia_min_pelota_m': 'min',  # El jugador más cerca
    'distancia_media_red_m': 'mean',
    'distancia_media_companero_m': 'mean',
    'num_golpes': 'sum',  # Suma de golpes del equipo
    'tiempo_promedio_desde_ultimo_golpe_s': 'mean',
    'aceleracion_media_mps2': 'mean',
    'aceleracion_std_mps2': 'mean',
    'duracion_total_s': 'mean',
    'frames_cerca_pelota': 'sum',
    'frames_zona_ofensiva': 'sum',
    'frames_zona_defensiva': 'sum',
    'total_frames': 'mean',
    'pct_frames_cerca_pelota': 'mean',
    'pct_tiempo_zona_ofensiva': 'mean',
    'tiempo_zona_defensiva_s': 'sum',
    
    # Datos demográficos (promedio del equipo)
    'EDAD': 'mean',
    'ESTATURA': 'mean',
    'TALLA': 'mean',
    
    # Contadores
    'player_name_normalized': lambda x: list(x)  # Lista de jugadores
}).reset_index()

# Renombrar columnas para claridad
df_equipos = df_equipos.rename(columns={
    'player_name_normalized': 'jugadores',
    'velocidad_media_mps': 'velocidad_media_equipo_mps',
    'distancia_total_m': 'distancia_total_equipo_m',
    'num_golpes': 'golpes_totales_equipo',
    'distancia_media_pelota_m': 'distancia_media_pelota_equipo_m',
    'pct_frames_cerca_pelota': 'pct_frames_cerca_pelota_equipo',
    'EDAD': 'edad_promedio_equipo',
    'ESTATURA': 'estatura_promedio_equipo',
    'TALLA': 'talla_promedio_equipo'
})

# Crear variable objetivo: si el equipo ganó el partido
df_equipos['gano_partido'] = (df_equipos['ganador'] == ('team_' + df_equipos['team_num'].astype(str))).astype(int)

# Calcular ventaja sobre equipo contrario para cada partido
partidos_completos = []
for partido in df_equipos['partido_num'].unique():
    equipos_partido = df_equipos[df_equipos['partido_num'] == partido].copy()
    
    if len(equipos_partido) == 2:  # Asegurar que hay 2 equipos
        team1 = equipos_partido[equipos_partido['team_num'] == 1].iloc[0]
        team2 = equipos_partido[equipos_partido['team_num'] == 2].iloc[0]
        
        # Añadir diferencias como features
        for idx, team in equipos_partido.iterrows():
            rival = team2 if team['team_num'] == 1 else team1
            
            df_equipos.loc[idx, 'diff_golpes_vs_rival'] = team['golpes_totales_equipo'] - rival['golpes_totales_equipo']
            df_equipos.loc[idx, 'diff_distancia_pelota_vs_rival'] = rival['distancia_media_pelota_equipo_m'] - team['distancia_media_pelota_equipo_m']  # Invertido: menor es mejor
            df_equipos.loc[idx, 'diff_pct_cerca_vs_rival'] = team['pct_frames_cerca_pelota_equipo'] - rival['pct_frames_cerca_pelota_equipo']
            df_equipos.loc[idx, 'diff_velocidad_vs_rival'] = team['velocidad_media_equipo_mps'] - rival['velocidad_media_equipo_mps']

print(f"\n✅ Dataset de equipos creado!")
print(f"\nDimensiones: {df_equipos.shape}")
print(f"Partidos únicos: {df_equipos['partido_num'].nunique()}")
print(f"Total registros (equipos-partido): {len(df_equipos)}")
print(f"Equipos ganadores: {df_equipos['gano_partido'].sum()} ({df_equipos['gano_partido'].mean():.1%})")

print(f"\nColumnas del dataset:")
print(df_equipos.columns.tolist())

CREACIÓN DE DATASET DE EQUIPOS

✅ Dataset de equipos creado!

Dimensiones: (54, 37)
Partidos únicos: 32
Total registros (equipos-partido): 54
Equipos ganadores: 27 (50.0%)

Columnas del dataset:
['partido_num', 'equipo', 'team_num', 'ganador', 'puntos_team_1', 'puntos_team_2', 'velocidad_media_equipo_mps', 'player_speed_mps_<lambda_0>', 'velocidad_std_mps', 'distancia_total_equipo_m', 'distancia_media_por_frame_m', 'desplazamiento_std_m', 'distancia_media_pelota_equipo_m', 'distancia_min_pelota_m', 'distancia_media_red_m', 'distancia_media_companero_m', 'golpes_totales_equipo', 'tiempo_promedio_desde_ultimo_golpe_s', 'aceleracion_media_mps2', 'aceleracion_std_mps2', 'duracion_total_s', 'frames_cerca_pelota', 'frames_zona_ofensiva', 'frames_zona_defensiva', 'total_frames', 'pct_frames_cerca_pelota_equipo', 'pct_tiempo_zona_ofensiva', 'tiempo_zona_defensiva_s', 'edad_promedio_equipo', 'estatura_promedio_equipo', 'talla_promedio_equipo', 'jugadores', 'gano_partido', 'diff_golpes_vs_rival'

In [ ]:
# Vista previa del dataset de equipos
print("="*80)
print("VISTA PREVIA - DATASET DE EQUIPOS")
print("="*80)

print("\n📊 Primeros 5 registros:")
display(df_equipos.head())

print("\n📊 Estadísticas descriptivas - Métricas clave:")
cols_resumen = [
    'velocidad_media_equipo_mps', 'distancia_total_equipo_m', 
    'golpes_totales_equipo', 'distancia_media_pelota_equipo_m',
    'pct_frames_cerca_pelota_equipo', 'diff_golpes_vs_rival',
    'diff_distancia_pelota_vs_rival', 'gano_partido'
]
display(df_equipos[cols_resumen].describe().round(3))

print("\n📊 Información del dataset:")
print(df_equipos.info())

VISTA PREVIA - DATASET DE EQUIPOS

📊 Primeros 5 registros:


,partido_num,equipo,team_num,ganador,puntos_team_1,puntos_team_2,velocidad_media_equipo_mps,player_speed_mps_<lambda_0>,velocidad_std_mps,distancia_total_equipo_m,...,tiempo_zona_defensiva_s,edad_promedio_equipo,estatura_promedio_equipo,talla_promedio_equipo,jugadores,gano_partido,diff_golpes_vs_rival,diff_distancia_pelota_vs_rival,diff_pct_cerca_vs_rival,diff_velocidad_vs_rival
0,1,match_1_team_2,2,team_1,7,1,0.477014,1.697056,0.950588,111.836009,...,0.0,19.0,176.5,74.5,"[carlos felipe andrade saenz, santiago navarro...",0,NaN,NaN,NaN,NaN
1,3,match_3_team_1,1,team_2,5,7,0.946091,3.394113,1.576347,418.041529,...,0.0,19.5,176.5,72.0,"[maria jose cifuentes, santiago urrego rodriguez]",0,-8.0,-3.681179,-21.979138,0.438126
2,3,match_3_team_2,2,team_2,5,7,0.507965,3.394113,1.231096,149.963206,...,0.0,19.0,173.0,68.0,[jonatan nicolas marmolejo lopez],1,8.0,3.681179,21.979138,-0.438126
3,4,match_4_team_1,1,team_1,7,2,1.074377,3.782056,1.636089,396.432346,...,0.0,19.0,175.5,71.0,"[luis jorge garcia camargo, rafael andres salc...",1,-17.0,0.028285,-12.333024,0.399168
4,4,match_4_team_2,2,team_1,7,2,0.675209,3.394113,1.393713,380.197174,...,0.0,19.5,178.0,76.0,"[juan pablo aponte veloza, santiago navarro cuy]",0,17.0,-0.028285,12.333024,-0.399168



📊 Estadísticas descriptivas - Métricas clave:


,velocidad_media_equipo_mps,distancia_total_equipo_m,golpes_totales_equipo,distancia_media_pelota_equipo_m,pct_frames_cerca_pelota_equipo,diff_golpes_vs_rival,diff_distancia_pelota_vs_rival,gano_partido
count,54.000,54.000,54.000,54.000,54.000,44.000,44.000,54.000
mean,0.710,393.929,54.889,6.610,9.919,0.000,0.000,0.500
std,0.263,212.265,71.245,2.154,9.079,107.322,3.661,0.505
min,0.314,98.033,0.000,3.217,0.000,-329.000,-10.775,0.000
25%,0.494,262.322,10.000,5.068,2.775,-36.250,-1.968,0.000
50%,0.670,364.047,29.500,5.813,7.359,0.000,0.000,0.500
75%,0.928,446.255,58.000,7.915,15.391,36.250,1.968,1.000
max,1.530,1064.677,353.000,13.992,29.053,329.000,10.775,1.000



📊 Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 37 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   partido_num                           54 non-null     int64  
 1   equipo                                54 non-null     object 
 2   team_num                              54 non-null     int64  
 3   ganador                               54 non-null     object 
 4   puntos_team_1                         54 non-null     int64  
 5   puntos_team_2                         54 non-null     int64  
 6   velocidad_media_equipo_mps            54 non-null     float64
 7   player_speed_mps_<lambda_0>           54 non-null     float64
 8   velocidad_std_mps                     54 non-null     float64
 9   distancia_total_equipo_m              54 non-null     float64
 10  distancia_media_por_frame_m           54 non-null     float6

In [ ]:
# Guardar dataset de equipos
output_equipos = DATA_DIR / 'processed' / 'Equipo_Partido_Features.csv'
df_equipos.to_csv(output_equipos, index=False)
print(f"\n💾 Dataset de equipos guardado en: {output_equipos}")
print(f"   Dimensiones: {df_equipos.shape}")
print(f"   Columnas: {df_equipos.shape[1]}")
print(f"   Registros: {df_equipos.shape[0]}")


💾 Dataset de equipos guardado en: Equipo_Partido_Features.csv
   Dimensiones: (54, 37)
   Columnas: 37
   Registros: 54
